In [1]:
# ==================================================================================================
# CISLR — L15
# FINAL 211-CLASS MODEL SELECTION + TEST-PROTOCOL FREEZE
# ==================================================================================================
#
# PURPOSE
# -------
# L15 is NOT a training stage.
#
# It formally freezes:
#
#   FINAL 211-CLASS RESEARCH SYSTEM:
#
#       L12B SSL ST-GCN               weight = 0.40
#       L13B RGB Temporal Transformer weight = 0.30
#       L14A VideoMAE Transfer        weight = 0.30
#
#       Fusion representation:
#           Sparse Top-5 raw-probability late fusion
#
#       Frozen validation result:
#           Top-1 = 41.232227% = 87 / 211
#           Top-5 = 51.658768% = 109 / 211
#
# L15 performs:
#
#   1. Verify all required frozen artifacts.
#   2. Verify L14C configuration.
#   3. Verify L14C validation result.
#   4. Verify selected branches and weights.
#   5. Hash all critical frozen artifacts.
#   6. Create immutable model-selection manifest.
#   7. Create final-test protocol for L16.
#   8. Create frozen leaderboard.
#   9. Create reproducibility/environment snapshot.
#  10. Create L16 authorization/protocol file.
#
# IMPORTANT:
#
#   L15 DOES NOT:
#
#       - open l9_test_manifest.csv
#       - decode test videos
#       - inspect test labels
#       - run test inference
#       - tune weights
#       - train any model
#       - modify L12B/L13B/L14A/L14C
#
# The test set remains locked until L16.
#
# ==================================================================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import os
import sys
import platform
import shutil
import subprocess
import pandas as pd
import numpy as np

try:
    import torch
except Exception:
    torch = None

try:
    import openvino as ov
except Exception:
    ov = None


# ==================================================================================================
# 0. CONFIGURATION
# ==================================================================================================

SEED = 42

NUM_CLASSES = 211
EXPECTED_VAL_SAMPLES = 211

EXPECTED_TOP1 = 41.232227
EXPECTED_TOP5 = 51.658768

EXPECTED_CORRECT_TOP1 = 87
EXPECTED_CORRECT_TOP5 = 109

EXPECTED_REPRESENTATION = "raw_probability"

EXPECTED_MODELS = [
    "L13B",
    "L14A",
    "L12B",
]

EXPECTED_WEIGHTS = {
    "L13B": 0.30,
    "L14A": 0.30,
    "L12B": 0.40,
}

TOLERANCE = 1e-5


ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/"
    "CISLR_LANDMARK_MODEL"
)


# --------------------------------------------------------------------------------------------------
# Frozen source artifacts
# --------------------------------------------------------------------------------------------------

L12B_CKPT = (
    ROOT /
    "audit/l12b_ssl_stgcn/models/"
    "l12b_best_stgcn.pt"
)

L12B_PRED = (
    ROOT /
    "audit/l12b_ssl_stgcn/reports/"
    "l12b_validation_predictions.csv"
)


L13B_CKPT = (
    ROOT /
    "audit/l13_multimodal/l13_b_rgb_temporal/models/"
    "l13b_best_rgb_temporal.pt"
)

L13B_PRED = (
    ROOT /
    "audit/l13_multimodal/l13_b_rgb_temporal/reports/"
    "l13b_validation_predictions.csv"
)


L14A_CKPT = (
    ROOT /
    "audit/l14_pretrained_video/l14a_videomae/models/"
    "l14a_best_videomae.pt"
)

L14A_PRED = (
    ROOT /
    "audit/l14_pretrained_video/l14a_videomae/reports/"
    "l14a_validation_predictions.csv"
)


L14C_ROOT = (
    ROOT /
    "audit/l14_pretrained_video/l14c_late_fusion"
)

L14C_CONFIG = (
    L14C_ROOT /
    "models/l14c_fusion_config.json"
)

L14C_SCORES = (
    L14C_ROOT /
    "models/l14c_validation_fusion_scores.npz"
)

L14C_PRED = (
    L14C_ROOT /
    "reports/l14c_validation_predictions.csv"
)

L14C_COMPARISON = (
    L14C_ROOT /
    "reports/l14c_model_comparison.csv"
)

L14C_SUMMARY = (
    L14C_ROOT /
    "reports/l14c_summary.json"
)


# --------------------------------------------------------------------------------------------------
# L15 output
# --------------------------------------------------------------------------------------------------

L15_ROOT = (
    ROOT /
    "audit/l15_final_model_selection"
)

MODEL_DIR = L15_ROOT / "models"
REPORT_DIR = L15_ROOT / "reports"
PROTOCOL_DIR = L15_ROOT / "protocol"
HASH_DIR = L15_ROOT / "hashes"

for p in [
    L15_ROOT,
    MODEL_DIR,
    REPORT_DIR,
    PROTOCOL_DIR,
    HASH_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )


print("=" * 132)
print("CISLR — L15")
print("FINAL 211-CLASS MODEL SELECTION + TEST-PROTOCOL FREEZE")
print("=" * 132)

print()
print("Research task       : 211-class controlled CISLR recognition")
print("Selection source    : validation only")
print("Selected stage      : L14C")
print("Classes             :", NUM_CLASSES)
print("Validation samples  :", EXPECTED_VAL_SAMPLES)
print("Seed                :", SEED)

print()
print("EXPECTED FINAL SYSTEM")
print("-" * 132)

print("L13B RGB Temporal Transformer : 0.30")
print("L14A VideoMAE Transfer        : 0.30")
print("L12B SSL ST-GCN               : 0.40")

print()
print("Fusion representation         : raw_probability")
print("Fusion type                   : sparse Top-5 late fusion")

print()
print("Expected validation Top-1     : 41.232227%")
print("Expected validation Top-5     : 51.658768%")
print("Expected correct Top-1        : 87/211")
print("Expected correct Top-5        : 109/211")

print()
print("TEST STATUS")
print("-" * 132)
print("Test manifest opened          : NO")
print("Test labels inspected         : NO")
print("Test videos decoded           : NO")
print("Test inference performed      : NO")


# ==================================================================================================
# 1. HELPER FUNCTIONS
# ==================================================================================================

def sha256_file(path, block_size=1024 * 1024):

    path = Path(path)

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(block_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def file_info(path):

    path = Path(path)

    return {
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": (
            int(path.stat().st_size)
            if path.exists()
            else None
        ),
        "sha256": (
            sha256_file(path)
            if path.exists()
            else None
        ),
    }


def save_json(obj, path):

    path = Path(path)

    with open(path, "w") as f:
        json.dump(
            obj,
            f,
            indent=2
        )


def almost_equal(a, b, tol=TOLERANCE):

    return abs(
        float(a) - float(b)
    ) <= tol


def passfail(condition):

    return (
        "PASS"
        if condition
        else "FAIL"
    )


# ==================================================================================================
# 2. REQUIRED ARTIFACT VERIFICATION
# ==================================================================================================

print("\n" + "=" * 132)
print("1. REQUIRED FROZEN ARTIFACT VERIFICATION")
print("=" * 132)

required_artifacts = {

    "L12B best checkpoint":
        L12B_CKPT,

    "L12B validation predictions":
        L12B_PRED,

    "L13B best checkpoint":
        L13B_CKPT,

    "L13B validation predictions":
        L13B_PRED,

    "L14A best checkpoint":
        L14A_CKPT,

    "L14A validation predictions":
        L14A_PRED,

    "L14C fusion config":
        L14C_CONFIG,

    "L14C validation scores":
        L14C_SCORES,

    "L14C validation predictions":
        L14C_PRED,

    "L14C comparison":
        L14C_COMPARISON,

    "L14C summary":
        L14C_SUMMARY,
}


artifact_status = {}

for name, path in required_artifacts.items():

    exists = path.exists()

    artifact_status[name] = exists

    print(
        f"{name:36s}: "
        f"{passfail(exists):4s} | "
        f"{path}"
    )

if not all(
    artifact_status.values()
):

    missing = [
        name
        for name, ok
        in artifact_status.items()
        if not ok
    ]

    raise FileNotFoundError(
        "Missing required frozen artifacts:\n"
        +
        "\n".join(missing)
    )

print()
print(
    "[PASS] All required frozen artifacts exist."
)


# ==================================================================================================
# 3. LOAD L14C CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 132)
print("2. VERIFYING L14C FROZEN CONFIGURATION")
print("=" * 132)

with open(
    L14C_CONFIG,
    "r"
) as f:

    l14c_config = json.load(f)


print(
    "Stage               :",
    l14c_config.get(
        "stage"
    )
)

print(
    "Method              :",
    l14c_config.get(
        "method"
    )
)

print(
    "Representation      :",
    l14c_config.get(
        "representation"
    )
)

print(
    "Models              :",
    l14c_config.get(
        "models"
    )
)

print(
    "Weights             :",
    l14c_config.get(
        "weights"
    )
)

print(
    "Validation Top-1    :",
    l14c_config.get(
        "validation_top1"
    )
)

print(
    "Validation Top-5    :",
    l14c_config.get(
        "validation_top5"
    )
)


# --------------------------------------------------------------------------------------------------
# Check representation
# --------------------------------------------------------------------------------------------------

actual_rep = (
    l14c_config.get(
        "representation"
    )
)

rep_ok = (
    actual_rep
    ==
    EXPECTED_REPRESENTATION
)

print()
print(
    f"Representation check : "
    f"{passfail(rep_ok)}"
)

if not rep_ok:

    raise RuntimeError(
        "L14C representation differs from frozen expectation."
    )


# --------------------------------------------------------------------------------------------------
# Check model set
# --------------------------------------------------------------------------------------------------

actual_models = (
    l14c_config.get(
        "models",
        []
    )
)

models_ok = (
    set(actual_models)
    ==
    set(EXPECTED_MODELS)
)

print(
    f"Model-set check      : "
    f"{passfail(models_ok)}"
)

if not models_ok:

    raise RuntimeError(
        f"L14C model set mismatch.\n"
        f"Expected: {EXPECTED_MODELS}\n"
        f"Actual:   {actual_models}"
    )


# --------------------------------------------------------------------------------------------------
# Check weights
# --------------------------------------------------------------------------------------------------

actual_weights = (
    l14c_config.get(
        "weights",
        {}
    )
)

weight_checks = {}

for model_name, expected_weight in (
    EXPECTED_WEIGHTS.items()
):

    actual_weight = (
        actual_weights.get(
            model_name,
            None
        )
    )

    ok = (
        actual_weight is not None
        and
        almost_equal(
            actual_weight,
            expected_weight
        )
    )

    weight_checks[
        model_name
    ] = ok

    print(
        f"{model_name:5s} weight check    : "
        f"{passfail(ok)} | "
        f"expected={expected_weight:.2f} "
        f"actual={actual_weight}"
    )

if not all(
    weight_checks.values()
):

    raise RuntimeError(
        "Frozen L14C fusion weights do not match expectation."
    )


weight_sum = sum(
    float(v)
    for v in actual_weights.values()
)

sum_ok = almost_equal(
    weight_sum,
    1.0
)

print(
    f"Weight-sum check     : "
    f"{passfail(sum_ok)} | "
    f"sum={weight_sum:.12f}"
)

if not sum_ok:

    raise RuntimeError(
        "L14C fusion weights do not sum to 1."
    )


# ==================================================================================================
# 4. VERIFY L14C SUMMARY
# ==================================================================================================

print("\n" + "=" * 132)
print("3. VERIFYING L14C VALIDATION RESULT")
print("=" * 132)

with open(
    L14C_SUMMARY,
    "r"
) as f:

    l14c_summary = json.load(f)


summary_top1 = float(
    l14c_summary[
        "val_top1"
    ]
)

summary_top5 = float(
    l14c_summary[
        "val_top5"
    ]
)

summary_correct1 = int(
    l14c_summary[
        "correct_top1"
    ]
)

summary_correct5 = int(
    l14c_summary[
        "correct_top5"
    ]
)


checks = {

    "Top-1":
        almost_equal(
            summary_top1,
            EXPECTED_TOP1
        ),

    "Top-5":
        almost_equal(
            summary_top5,
            EXPECTED_TOP5
        ),

    "Correct Top-1":
        summary_correct1
        ==
        EXPECTED_CORRECT_TOP1,

    "Correct Top-5":
        summary_correct5
        ==
        EXPECTED_CORRECT_TOP5,
}


print(
    f"Top-1       : "
    f"{summary_top1:.6f}% | "
    f"{passfail(checks['Top-1'])}"
)

print(
    f"Top-5       : "
    f"{summary_top5:.6f}% | "
    f"{passfail(checks['Top-5'])}"
)

print(
    f"Correct T1  : "
    f"{summary_correct1}/211 | "
    f"{passfail(checks['Correct Top-1'])}"
)

print(
    f"Correct T5  : "
    f"{summary_correct5}/211 | "
    f"{passfail(checks['Correct Top-5'])}"
)


if not all(
    checks.values()
):

    raise RuntimeError(
        "L14C frozen validation metrics differ from expected values."
    )

print()
print(
    "[PASS] Frozen L14C validation result verified."
)


# ==================================================================================================
# 5. VERIFY L14C SCORE FILE
# ==================================================================================================

print("\n" + "=" * 132)
print("4. VERIFYING L14C SCORE MATRIX")
print("=" * 132)

score_data = np.load(
    L14C_SCORES,
    allow_pickle=False
)

print(
    "Keys:",
    list(
        score_data.files
    )
)

required_score_keys = {

    "uid",
    "targets",
    "fusion_scores",
    "predictions",
    "top5",

}

missing_score_keys = (
    required_score_keys
    -
    set(score_data.files)
)

if missing_score_keys:

    raise RuntimeError(
        "L14C score file missing keys: "
        +
        str(
            missing_score_keys
        )
    )


score_uids = (
    score_data[
        "uid"
    ]
)

score_targets = (
    score_data[
        "targets"
    ]
)

score_matrix = (
    score_data[
        "fusion_scores"
    ]
)

score_predictions = (
    score_data[
        "predictions"
    ]
)

score_top5 = (
    score_data[
        "top5"
    ]
)


print(
    "UID shape          :",
    score_uids.shape
)

print(
    "Target shape       :",
    score_targets.shape
)

print(
    "Score shape        :",
    score_matrix.shape
)

print(
    "Prediction shape   :",
    score_predictions.shape
)

print(
    "Top-5 shape        :",
    score_top5.shape
)


assert score_uids.shape == (
    EXPECTED_VAL_SAMPLES,
)

assert score_targets.shape == (
    EXPECTED_VAL_SAMPLES,
)

assert score_matrix.shape == (
    EXPECTED_VAL_SAMPLES,
    NUM_CLASSES,
)

assert score_predictions.shape == (
    EXPECTED_VAL_SAMPLES,
)

assert score_top5.shape == (
    EXPECTED_VAL_SAMPLES,
    5,
)

assert np.isfinite(
    score_matrix
).all()

print()
print(
    "[PASS] Frozen L14C score matrix is structurally valid."
)


# ==================================================================================================
# 6. INDEPENDENTLY RECOMPUTE L14C METRICS
# ==================================================================================================

print("\n" + "=" * 132)
print("5. INDEPENDENT METRIC RECOMPUTATION")
print("=" * 132)

recomputed_pred = np.argmax(
    score_matrix,
    axis=1
)

recomputed_order = np.argsort(
    -score_matrix,
    axis=1
)

recomputed_top5 = (
    recomputed_order[
        :,
        :5
    ]
)

correct1_mask = (
    recomputed_pred
    ==
    score_targets
)

correct5_mask = np.array(
    [
        int(
            score_targets[i]
        )
        in
        recomputed_top5[i]

        for i in range(
            EXPECTED_VAL_SAMPLES
        )
    ],
    dtype=bool
)


recomputed_correct1 = int(
    correct1_mask.sum()
)

recomputed_correct5 = int(
    correct5_mask.sum()
)

recomputed_top1 = (
    recomputed_correct1
    /
    EXPECTED_VAL_SAMPLES
    *
    100.0
)

recomputed_top5_acc = (
    recomputed_correct5
    /
    EXPECTED_VAL_SAMPLES
    *
    100.0
)


print(
    f"Recomputed Top-1 : "
    f"{recomputed_top1:.6f}% "
    f"({recomputed_correct1}/211)"
)

print(
    f"Recomputed Top-5 : "
    f"{recomputed_top5_acc:.6f}% "
    f"({recomputed_correct5}/211)"
)


metric_recompute_ok = (

    recomputed_correct1
    ==
    EXPECTED_CORRECT_TOP1

    and

    recomputed_correct5
    ==
    EXPECTED_CORRECT_TOP5

    and

    almost_equal(
        recomputed_top1,
        EXPECTED_TOP1
    )

    and

    almost_equal(
        recomputed_top5_acc,
        EXPECTED_TOP5
    )
)

print()
print(
    "Metric recomputation:",
    passfail(
        metric_recompute_ok
    )
)

if not metric_recompute_ok:

    raise RuntimeError(
        "Independent L14C metric recomputation failed."
    )


# ==================================================================================================
# 7. VERIFY PREDICTION REPORT
# ==================================================================================================

print("\n" + "=" * 132)
print("6. VERIFYING L14C VALIDATION PREDICTION REPORT")
print("=" * 132)

pred_df = pd.read_csv(
    L14C_PRED
)

print(
    "Rows    :",
    len(pred_df)
)

print(
    "Columns :",
    len(pred_df.columns)
)


if len(pred_df) != EXPECTED_VAL_SAMPLES:

    raise RuntimeError(
        "L14C prediction report does not contain 211 validation rows."
    )


if (
    pred_df["uid"]
    .astype(str)
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate validation UID detected in L14C prediction report."
    )


csv_correct1 = int(
    pred_df[
        "top1_correct"
    ]
    .astype(bool)
    .sum()
)

csv_correct5 = int(
    pred_df[
        "top5_correct"
    ]
    .astype(bool)
    .sum()
)


print(
    "CSV Top-1 correct :",
    csv_correct1
)

print(
    "CSV Top-5 correct :",
    csv_correct5
)


if (
    csv_correct1
    !=
    EXPECTED_CORRECT_TOP1
):

    raise RuntimeError(
        "L14C CSV Top-1 count mismatch."
    )


if (
    csv_correct5
    !=
    EXPECTED_CORRECT_TOP5
):

    raise RuntimeError(
        "L14C CSV Top-5 count mismatch."
    )


print()
print(
    "[PASS] L14C validation prediction report verified."
)


# ==================================================================================================
# 8. LOAD AND VERIFY FROZEN LEADERBOARD
# ==================================================================================================

print("\n" + "=" * 132)
print("7. FINAL MODEL-SELECTION LEADERBOARD")
print("=" * 132)

comparison_df = pd.read_csv(
    L14C_COMPARISON
)

print(
    comparison_df.to_string(
        index=False
    )
)


l14c_rows = (
    comparison_df[
        comparison_df[
            "stage"
        ]
        ==
        "L14C"
    ]
)


if len(l14c_rows) != 1:

    raise RuntimeError(
        "Expected exactly one L14C leaderboard row."
    )


leader_top1 = float(
    l14c_rows.iloc[0][
        "val_top1"
    ]
)

leader_top5 = float(
    l14c_rows.iloc[0][
        "val_top5"
    ]
)


if not almost_equal(
    leader_top1,
    EXPECTED_TOP1
):

    raise RuntimeError(
        "Leaderboard L14C Top-1 mismatch."
    )


if not almost_equal(
    leader_top5,
    EXPECTED_TOP5
):

    raise RuntimeError(
        "Leaderboard L14C Top-5 mismatch."
    )


print()
print(
    "[PASS] L14C leaderboard entry verified."
)


# ==================================================================================================
# 9. HASH ALL CRITICAL ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 132)
print("8. CRYPTOGRAPHIC ARTIFACT FREEZE")
print("=" * 132)

critical_files = {

    "l12b_checkpoint":
        L12B_CKPT,

    "l12b_validation_predictions":
        L12B_PRED,

    "l13b_checkpoint":
        L13B_CKPT,

    "l13b_validation_predictions":
        L13B_PRED,

    "l14a_checkpoint":
        L14A_CKPT,

    "l14a_validation_predictions":
        L14A_PRED,

    "l14c_fusion_config":
        L14C_CONFIG,

    "l14c_validation_scores":
        L14C_SCORES,

    "l14c_validation_predictions":
        L14C_PRED,

    "l14c_model_comparison":
        L14C_COMPARISON,

    "l14c_summary":
        L14C_SUMMARY,

}


hash_manifest = {}

for name, path in (
    critical_files.items()
):

    info = file_info(
        path
    )

    hash_manifest[
        name
    ] = info

    print(
        f"{name:34s} | "
        f"{info['sha256']}"
    )


HASH_MANIFEST_PATH = (
    HASH_DIR /
    "l15_artifact_sha256_manifest.json"
)

save_json(
    hash_manifest,
    HASH_MANIFEST_PATH
)

print()
print(
    "[PASS] SHA256 manifest saved:"
)

print(
    HASH_MANIFEST_PATH
)


# ==================================================================================================
# 10. ENVIRONMENT SNAPSHOT
# ==================================================================================================

print("\n" + "=" * 132)
print("9. REPRODUCIBILITY ENVIRONMENT SNAPSHOT")
print("=" * 132)

environment = {

    "timestamp":
        datetime.now().isoformat(),

    "python_executable":
        sys.executable,

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "working_directory":
        os.getcwd(),

    "seed":
        SEED,

    "numpy_version":
        np.__version__,

    "pandas_version":
        pd.__version__,

}


if torch is not None:

    environment[
        "torch_version"
    ] = torch.__version__

    environment[
        "cuda_available"
    ] = bool(
        torch.cuda.is_available()
    )

    if torch.cuda.is_available():

        environment[
            "cuda_device"
        ] = torch.cuda.get_device_name(
            0
        )

        environment[
            "cuda_device_count"
        ] = torch.cuda.device_count()


if ov is not None:

    environment[
        "openvino_version"
    ] = getattr(
        ov,
        "__version__",
        "unknown"
    )


ENV_PATH = (
    REPORT_DIR /
    "l15_environment_snapshot.json"
)

save_json(
    environment,
    ENV_PATH
)


for k, v in environment.items():

    text = str(v)

    if len(text) > 150:
        text = text[:150] + "..."

    print(
        f"{k:24s}: {text}"
    )


# ==================================================================================================
# 11. CREATE FROZEN MODEL-SELECTION MANIFEST
# ==================================================================================================

print("\n" + "=" * 132)
print("10. CREATING FINAL MODEL-SELECTION MANIFEST")
print("=" * 132)

selection_manifest = {

    "stage":
        "L15",

    "purpose":
        (
            "Final 211-class model selection "
            "before one-time test evaluation."
        ),

    "selection_status":
        "FROZEN",

    "selected_system":
        "L14C",

    "selected_method":
        (
            "Frozen Sparse Top-5 "
            "Late Fusion"
        ),

    "fusion_representation":
        EXPECTED_REPRESENTATION,

    "fusion_models":
        EXPECTED_MODELS,

    "fusion_weights":
        EXPECTED_WEIGHTS,

    "num_classes":
        NUM_CLASSES,

    "validation_samples":
        EXPECTED_VAL_SAMPLES,

    "validation_top1":
        EXPECTED_TOP1,

    "validation_top5":
        EXPECTED_TOP5,

    "validation_correct_top1":
        EXPECTED_CORRECT_TOP1,

    "validation_correct_top5":
        EXPECTED_CORRECT_TOP5,

    "selection_metric":
        (
            "Validation Top-1 accuracy; "
            "Top-5 used as secondary metric."
        ),

    "frozen_components": {

        "L12B": {
            "role":
                "landmark/skeleton branch",

            "model":
                "SSL ST-GCN",

            "weight":
                0.40,

            "checkpoint":
                str(
                    L12B_CKPT
                ),

            "checkpoint_sha256":
                hash_manifest[
                    "l12b_checkpoint"
                ][
                    "sha256"
                ],
        },

        "L13B": {
            "role":
                "RGB temporal branch",

            "model":
                "RGB Temporal Transformer",

            "weight":
                0.30,

            "checkpoint":
                str(
                    L13B_CKPT
                ),

            "checkpoint_sha256":
                hash_manifest[
                    "l13b_checkpoint"
                ][
                    "sha256"
                ],
        },

        "L14A": {
            "role":
                "pretrained video branch",

            "model":
                "VideoMAE Transfer",

            "weight":
                0.30,

            "checkpoint":
                str(
                    L14A_CKPT
                ),

            "checkpoint_sha256":
                hash_manifest[
                    "l14a_checkpoint"
                ][
                    "sha256"
                ],
        },

    },

    "important_methodological_limitation":
        (
            "L14C uses sparse Top-5 probabilities "
            "stored by the frozen component models. "
            "It is not full 211-class logit fusion."
        ),

    "test_status_at_freeze": {

        "test_manifest_opened":
            False,

        "test_labels_inspected":
            False,

        "test_video_decoded":
            False,

        "test_inference_performed":
            False,

        "test_used_for_model_selection":
            False,

    },

    "frozen_at":
        datetime.now().isoformat(),

}


SELECTION_MANIFEST_PATH = (
    MODEL_DIR /
    "l15_final_model_selection.json"
)

save_json(
    selection_manifest,
    SELECTION_MANIFEST_PATH
)

print(
    "[PASS] Saved:"
)

print(
    SELECTION_MANIFEST_PATH
)


# ==================================================================================================
# 12. CREATE L16 TEST PROTOCOL
# ==================================================================================================

print("\n" + "=" * 132)
print("11. CREATING ONE-TIME L16 TEST PROTOCOL")
print("=" * 132)

test_protocol = {

    "protocol_name":
        (
            "CISLR 211-Class "
            "Final Test Protocol"
        ),

    "created_in_stage":
        "L15",

    "execution_stage":
        "L16",

    "status":
        "FROZEN_BEFORE_TEST",

    "test_evaluation_count_allowed":
        1,

    "selected_system":
        "L14C",

    "num_classes":
        211,

    "frozen_fusion": {

        "representation":
            "raw_probability",

        "method":
            (
                "Sparse Top-5 "
                "late fusion"
            ),

        "weights": {

            "L12B": 0.40,
            "L13B": 0.30,
            "L14A": 0.30,

        },

    },

    "rules": [

        (
            "The L9 test partition may be opened "
            "only after this L15 protocol is frozen."
        ),

        (
            "No model parameters may be trained "
            "or fine-tuned using test data."
        ),

        (
            "Fusion weights must remain exactly "
            "L12B=0.40, L13B=0.30, L14A=0.30."
        ),

        (
            "No confidence threshold may be selected "
            "using test labels."
        ),

        (
            "No architecture choice may be changed "
            "after observing test results."
        ),

        (
            "No sample may be removed because it is "
            "misclassified or difficult."
        ),

        (
            "All 211 frozen test samples must be "
            "included in final evaluation."
        ),

        (
            "Report Top-1 and Top-5 accuracy "
            "regardless of result."
        ),

        (
            "The test result is a final evaluation, "
            "not a new validation score."
        ),

        (
            "If an implementation failure occurs, "
            "record the failure before rerunning. "
            "Do not modify model-selection decisions "
            "based on observed test labels."
        ),

    ],

    "required_test_outputs": [

        "per-sample predictions",
        "Top-1 accuracy",
        "Top-5 accuracy",
        "correct Top-1 count",
        "correct Top-5 count",
        "confusion matrix",
        "branch predictions",
        "fusion prediction",
        "runtime information",
        "artifact hashes",

    ],

    "forbidden_during_l16": [

        "training",
        "fine-tuning",
        "fusion-weight search",
        "architecture search",
        "threshold tuning on test labels",
        "sample exclusion",
        "class exclusion",
        "repeated test-driven optimization",

    ],

}


TEST_PROTOCOL_PATH = (
    PROTOCOL_DIR /
    "l15_l16_one_time_test_protocol.json"
)

save_json(
    test_protocol,
    TEST_PROTOCOL_PATH
)

print(
    "[PASS] Test protocol saved:"
)

print(
    TEST_PROTOCOL_PATH
)


# ==================================================================================================
# 13. CREATE HUMAN-READABLE PROTOCOL TEXT
# ==================================================================================================

protocol_text = f"""
CISLR — L15 FINAL MODEL SELECTION
=================================

STATUS
------
FROZEN BEFORE TEST

FINAL 211-CLASS SYSTEM
----------------------
Stage:
L14C

Method:
Frozen Sparse Top-5 Late Fusion

Component models:
L12B SSL ST-GCN               = 0.40
L13B RGB Temporal Transformer = 0.30
L14A VideoMAE Transfer        = 0.30

Fusion representation:
raw_probability

IMPORTANT:
The fusion uses the stored sparse Top-5 probabilities.
It is NOT full 211-class logit fusion.


VALIDATION RESULT
-----------------
Samples : 211
Classes : 211

Top-1   : {EXPECTED_TOP1:.6f}%
Correct : {EXPECTED_CORRECT_TOP1}/211

Top-5   : {EXPECTED_TOP5:.6f}%
Correct : {EXPECTED_CORRECT_TOP5}/211


TEST STATUS AT L15 FREEZE
-------------------------
Test manifest opened     : NO
Test labels inspected    : NO
Test videos decoded      : NO
Test inference performed : NO
Test used for selection  : NO


L16 RULE
--------
The frozen L14C system is evaluated on the L9 test partition once.

The following are frozen before opening the test set:

1. Model architectures
2. Model checkpoints
3. Class mapping
4. Preprocessing
5. Fusion representation
6. Fusion weights
7. Evaluation metrics
8. Sample population

No model selection may be performed using L16 results.


FROZEN FUSION
-------------
Final score:

S_final(c) =
    0.40 * S_L12B(c)
  + 0.30 * S_L13B(c)
  + 0.30 * S_L14A(c)


FINAL TEST REPORT
-----------------
L16 must report:

- Top-1 accuracy
- Top-5 accuracy
- Correct Top-1 count
- Correct Top-5 count
- Per-sample predictions
- Confusion matrix
- Individual branch predictions
- Final fused predictions

The result must be reported regardless of whether it is higher
or lower than validation accuracy.


SCIENTIFIC SEPARATION
---------------------
This is the 211-class research benchmark.

Any later restricted-vocabulary deployment experiment
(e.g. 10-sign high-reliability system) must be reported
separately and must not replace or overwrite this benchmark.
"""


PROTOCOL_TXT_PATH = (
    PROTOCOL_DIR /
    "L15_FINAL_TEST_PROTOCOL.txt"
)

with open(
    PROTOCOL_TXT_PATH,
    "w"
) as f:

    f.write(
        protocol_text.strip()
        +
        "\n"
    )

print(
    "[PASS] Human-readable protocol saved:"
)

print(
    PROTOCOL_TXT_PATH
)


# ==================================================================================================
# 14. SAVE FINAL FROZEN LEADERBOARD
# ==================================================================================================

print("\n" + "=" * 132)
print("12. FREEZING VALIDATION LEADERBOARD")
print("=" * 132)

leaderboard = comparison_df.copy()

leaderboard[
    "selected_for_final_test"
] = (
    leaderboard[
        "stage"
    ]
    ==
    "L14C"
)

leaderboard[
    "test_evaluated"
] = False


LEADERBOARD_PATH = (
    REPORT_DIR /
    "l15_frozen_validation_leaderboard.csv"
)

leaderboard.to_csv(
    LEADERBOARD_PATH,
    index=False
)

print(
    leaderboard.to_string(
        index=False
    )
)

print()
print(
    "[PASS] Leaderboard saved:"
)

print(
    LEADERBOARD_PATH
)


# ==================================================================================================
# 15. CREATE FINAL SELECTION REPORT
# ==================================================================================================

print("\n" + "=" * 132)
print("13. FINAL SELECTION REPORT")
print("=" * 132)

selection_report = {

    "stage":
        "L15",

    "decision":
        (
            "L14C selected as final "
            "211-class research system."
        ),

    "reason":
        (
            "Highest frozen validation Top-1 "
            "accuracy among evaluated systems."
        ),

    "selected_validation_top1":
        EXPECTED_TOP1,

    "selected_validation_top5":
        EXPECTED_TOP5,

    "selected_correct_top1":
        EXPECTED_CORRECT_TOP1,

    "selected_correct_top5":
        EXPECTED_CORRECT_TOP5,

    "previous_best_top1_model":
        "L13B",

    "previous_best_top1":
        38.862559,

    "improvement_over_l13b_pp":
        float(
            EXPECTED_TOP1
            -
            38.862559
        ),

    "selected_components": {

        "L12B": 0.40,
        "L13B": 0.30,
        "L14A": 0.30,

    },

    "test_evaluated":
        False,

    "next_stage":
        (
            "L16 — one-time final "
            "211-class test evaluation"
        ),

}


SELECTION_REPORT_PATH = (
    REPORT_DIR /
    "l15_selection_report.json"
)

save_json(
    selection_report,
    SELECTION_REPORT_PATH
)

print(
    json.dumps(
        selection_report,
        indent=2
    )
)


# ==================================================================================================
# 16. COPY ONLY SMALL CONFIG/REPORT ARTIFACTS INTO L15 FREEZE
# ==================================================================================================

print("\n" + "=" * 132)
print("14. COPYING SMALL FREEZE ARTIFACTS")
print("=" * 132)

# Do NOT duplicate large neural-network checkpoints.
# Their absolute paths + SHA256 hashes are frozen instead.

copies = {

    L14C_CONFIG:
        MODEL_DIR /
        "source_l14c_fusion_config.json",

    L14C_SUMMARY:
        REPORT_DIR /
        "source_l14c_summary.json",

    L14C_COMPARISON:
        REPORT_DIR /
        "source_l14c_model_comparison.csv",

}


for src, dst in copies.items():

    shutil.copy2(
        src,
        dst
    )

    print(
        "[PASS]",
        dst
    )


# ==================================================================================================
# 17. HASH L15'S OWN FREEZE FILES
# ==================================================================================================

print("\n" + "=" * 132)
print("15. HASHING L15 FREEZE FILES")
print("=" * 132)

l15_files_to_hash = [

    SELECTION_MANIFEST_PATH,
    TEST_PROTOCOL_PATH,
    PROTOCOL_TXT_PATH,
    LEADERBOARD_PATH,
    SELECTION_REPORT_PATH,
    ENV_PATH,
    HASH_MANIFEST_PATH,

]

l15_hashes = {}

for path in (
    l15_files_to_hash
):

    h = sha256_file(
        path
    )

    l15_hashes[
        path.name
    ] = h

    print(
        f"{path.name:45s} "
        f"{h}"
    )


L15_HASH_PATH = (
    HASH_DIR /
    "l15_freeze_files_sha256.json"
)

save_json(
    l15_hashes,
    L15_HASH_PATH
)


# ==================================================================================================
# 18. MASTER FREEZE RECORD
# ==================================================================================================

print("\n" + "=" * 132)
print("16. CREATING MASTER FREEZE RECORD")
print("=" * 132)

master_freeze = {

    "stage":
        "L15",

    "status":
        "COMPLETE_AND_FROZEN",

    "timestamp":
        datetime.now().isoformat(),

    "project":
        "CISLR Research",

    "task":
        (
            "211-class isolated "
            "Indian Sign Language recognition"
        ),

    "selected_system":
        "L14C",

    "selected_method":
        (
            "Frozen Sparse Top-5 "
            "Late Fusion"
        ),

    "validation": {

        "samples":
            211,

        "classes":
            211,

        "top1":
            EXPECTED_TOP1,

        "top5":
            EXPECTED_TOP5,

        "correct_top1":
            EXPECTED_CORRECT_TOP1,

        "correct_top5":
            EXPECTED_CORRECT_TOP5,

    },

    "fusion": {

        "L12B":
            0.40,

        "L13B":
            0.30,

        "L14A":
            0.30,

    },

    "test_lock": {

        "opened":
            False,

        "decoded":
            False,

        "evaluated":
            False,

        "used_for_selection":
            False,

    },

    "next_stage":
        "L16",

    "next_stage_description":
        (
            "One-time final evaluation "
            "on the frozen L9 test partition."
        ),

    "source_artifact_hash_manifest":
        str(
            HASH_MANIFEST_PATH
        ),

    "selection_manifest":
        str(
            SELECTION_MANIFEST_PATH
        ),

    "test_protocol":
        str(
            TEST_PROTOCOL_PATH
        ),

    "leaderboard":
        str(
            LEADERBOARD_PATH
        ),

}


MASTER_FREEZE_PATH = (
    L15_ROOT /
    "L15_MASTER_FREEZE.json"
)

save_json(
    master_freeze,
    MASTER_FREEZE_PATH
)


MASTER_FREEZE_SHA256 = (
    sha256_file(
        MASTER_FREEZE_PATH
    )
)


print(
    "[PASS] Master freeze record:"
)

print(
    MASTER_FREEZE_PATH
)

print(
    "SHA256:",
    MASTER_FREEZE_SHA256
)


# ==================================================================================================
# 19. FINAL SAFETY ASSERTIONS
# ==================================================================================================

print("\n" + "=" * 132)
print("17. FINAL SAFETY ASSERTIONS")
print("=" * 132)

# We intentionally never construct/read the L9 test-manifest path in this cell.
# No test dataframe exists.
# No test dataset exists.
# No test DataLoader exists.
# No test video is opened.

safety_checks = {

    "all_required_artifacts":
        all(
            artifact_status.values()
        ),

    "representation_frozen":
        rep_ok,

    "model_set_frozen":
        models_ok,

    "weights_frozen":
        all(
            weight_checks.values()
        ),

    "weights_sum_to_one":
        sum_ok,

    "validation_metrics_verified":
        all(
            checks.values()
        ),

    "metric_recomputation_verified":
        metric_recompute_ok,

    "score_matrix_shape":
        score_matrix.shape
        ==
        (
            211,
            211
        ),

    "validation_sample_count":
        len(
            pred_df
        )
        ==
        211,

    "test_manifest_opened":
        False,

    "test_video_decoded":
        False,

    "test_inference_performed":
        False,

    "test_used_for_selection":
        False,

}


for key, value in (
    safety_checks.items()
):

    # For test actions, False is the expected safe state.
    if key in {
        "test_manifest_opened",
        "test_video_decoded",
        "test_inference_performed",
        "test_used_for_selection",
    }:

        ok = (
            value is False
        )

    else:

        ok = bool(
            value
        )

    print(
        f"{key:35s}: "
        f"{passfail(ok)}"
    )

    if not ok:

        raise RuntimeError(
            f"L15 safety check failed: {key}"
        )


# ==================================================================================================
# 20. FINAL FREEZE OUTPUT
# ==================================================================================================

print("\n")
print("=" * 132)
print("L15 — FINAL 211-CLASS MODEL SELECTION FREEZE")
print("=" * 132)

print()
print(
    "SELECTED SYSTEM        : L14C"
)

print(
    "METHOD                 : Frozen Sparse Top-5 Late Fusion"
)

print(
    "REPRESENTATION         : raw_probability"
)

print()

print(
    "L12B WEIGHT            : 0.40"
)

print(
    "L13B WEIGHT            : 0.30"
)

print(
    "L14A WEIGHT            : 0.30"
)

print()

print(
    f"VALIDATION TOP-1       : "
    f"{EXPECTED_TOP1:.6f}%"
)

print(
    f"VALIDATION TOP-5       : "
    f"{EXPECTED_TOP5:.6f}%"
)

print(
    f"CORRECT TOP-1          : "
    f"{EXPECTED_CORRECT_TOP1}/211"
)

print(
    f"CORRECT TOP-5          : "
    f"{EXPECTED_CORRECT_TOP5}/211"
)

print()

print("-" * 132)

print(
    "TEST MANIFEST OPENED   : NO"
)

print(
    "TEST LABELS INSPECTED  : NO"
)

print(
    "TEST VIDEO LOADED      : NO"
)

print(
    "TEST INFERENCE         : NO"
)

print(
    "TEST USED FOR SELECTION: NO"
)

print("-" * 132)

print()

print(
    "Selection manifest     :",
    SELECTION_MANIFEST_PATH
)

print(
    "Test protocol          :",
    TEST_PROTOCOL_PATH
)

print(
    "Protocol text          :",
    PROTOCOL_TXT_PATH
)

print(
    "Artifact hashes        :",
    HASH_MANIFEST_PATH
)

print(
    "Frozen leaderboard     :",
    LEADERBOARD_PATH
)

print(
    "Selection report       :",
    SELECTION_REPORT_PATH
)

print(
    "Environment snapshot   :",
    ENV_PATH
)

print(
    "Master freeze          :",
    MASTER_FREEZE_PATH
)

print(
    "Master freeze SHA256   :",
    MASTER_FREEZE_SHA256
)

print()

print(
    "[PASS] L15 COMPLETE AND FROZEN"
)

print(
    "[LOCK] MODEL SELECTION IS NOW CLOSED"
)

print(
    "[NEXT] L16 = ONE-TIME FINAL 211-CLASS TEST EVALUATION"
)

print("=" * 132)

CISLR — L15
FINAL 211-CLASS MODEL SELECTION + TEST-PROTOCOL FREEZE

Research task       : 211-class controlled CISLR recognition
Selection source    : validation only
Selected stage      : L14C
Classes             : 211
Validation samples  : 211
Seed                : 42

EXPECTED FINAL SYSTEM
------------------------------------------------------------------------------------------------------------------------------------
L13B RGB Temporal Transformer : 0.30
L14A VideoMAE Transfer        : 0.30
L12B SSL ST-GCN               : 0.40

Fusion representation         : raw_probability
Fusion type                   : sparse Top-5 late fusion

Expected validation Top-1     : 41.232227%
Expected validation Top-5     : 51.658768%
Expected correct Top-1        : 87/211
Expected correct Top-5        : 109/211

TEST STATUS
------------------------------------------------------------------------------------------------------------------------------------
Test manifest opened          : NO
Test labe

In [2]:
# ==================================================================================================
# CISLR — L16
# ONE-TIME FINAL 211-CLASS TEST EVALUATION
# ==================================================================================================
#
# FROZEN BY L15
# -------------
# L12B SSL ST-GCN               weight = 0.40
# L13B RGB Temporal Transformer weight = 0.30
# L14A VideoMAE                  weight = 0.30
#
# Fusion:
#   Sparse Top-5 RAW probability fusion
#
# THIS CELL:
#   1. Verifies L15 freeze.
#   2. Verifies checkpoint SHA256 hashes.
#   3. Opens L9 test manifest for the FIRST final evaluation.
#   4. Runs frozen L12B on all 211 test samples.
#   5. Runs frozen L13B on all 211 test samples.
#   6. Runs frozen L14A on all 211 test samples.
#   7. Converts each branch to the SAME sparse Top-5 representation used by L14C.
#   8. Applies EXACT frozen weights 0.40 / 0.30 / 0.30.
#   9. Computes branch + fused Top-1 / Top-5.
#  10. Saves predictions, confusion matrix, scores, timings and final report.
#
# NO TRAINING.
# NO WEIGHT SEARCH.
# NO THRESHOLD TUNING.
# NO SAMPLE REMOVAL.
# NO MODEL SELECTION.
#
# ==================================================================================================

# ==================================================================================================
# 0. IMPORTS
# ==================================================================================================

import os
import gc
import cv2
import json
import time
import math
import random
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision

from torchvision import transforms
from torchvision.models import mobilenet_v3_small

from transformers import (
    VideoMAEForVideoClassification,
    VideoMAEImageProcessor,
)

warnings.filterwarnings("ignore")


# ==================================================================================================
# 1. REPRODUCIBILITY
# ==================================================================================================

SEED = 42


def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = DEVICE.type == "cuda"


# ==================================================================================================
# 2. FROZEN CONSTANTS
# ==================================================================================================

NUM_CLASSES = 211

SEQ_LEN = 32
NUM_NODES = 42
JOINTS_PER_HAND = 21
INPUT_CHANNELS = 11

L12_DROPOUT = 0.35

VISUAL_DIM = 576
MODEL_DIM = 256
NUM_HEADS = 4
NUM_LAYERS = 2
FF_DIM = 512
L13_DROPOUT = 0.35

IMAGE_SIZE = 224

CONTEXT_BOX_SCALE = 1.50
MIN_CONTEXT_BOX = 96
EMBED_BATCH_SIZE = 64

VIDEOMAE_FRAMES = 16

W_L12B = 0.40
W_L13B = 0.30
W_L14A = 0.30

assert abs(
    W_L12B +
    W_L13B +
    W_L14A -
    1.0
) < 1e-12


# ==================================================================================================
# 3. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)

VIDEO_DIR = Path(
    "/home/dipshikha/Music/cao/CISLR_v1.5-a_videos"
)

RAW_DIR = (
    ROOT /
    "audit/l6_full_extraction/raw_sequences"
)

L9_ROOT = (
    ROOT /
    "audit/l9_split_freeze"
)

TEST_MANIFEST = (
    L9_ROOT /
    "splits/l9_test_manifest.csv"
)

CLASS_MAPPING = (
    L9_ROOT /
    "class_mapping/l9_controlled_class_mapping.json"
)


# --------------------------------------------------------------------------------------------------
# L12B
# --------------------------------------------------------------------------------------------------

L12B_CKPT = (
    ROOT /
    "audit/l12b_ssl_stgcn/models/"
    "l12b_best_stgcn.pt"
)


# --------------------------------------------------------------------------------------------------
# L13A / L13B
# --------------------------------------------------------------------------------------------------

L13A_CKPT = (
    ROOT /
    "audit/l13_multimodal/l13_a_visual/models/"
    "l13a_best_visual_encoder.pt"
)

L13B_CKPT = (
    ROOT /
    "audit/l13_multimodal/l13_b_rgb_temporal/models/"
    "l13b_best_rgb_temporal.pt"
)


# --------------------------------------------------------------------------------------------------
# L14A
# --------------------------------------------------------------------------------------------------

L14A_CKPT = (
    ROOT /
    "audit/l14_pretrained_video/l14a_videomae/models/"
    "l14a_best_videomae.pt"
)


# --------------------------------------------------------------------------------------------------
# L15
# --------------------------------------------------------------------------------------------------

L15_ROOT = (
    ROOT /
    "audit/l15_final_model_selection"
)

L15_MASTER = (
    L15_ROOT /
    "L15_MASTER_FREEZE.json"
)

L15_SELECTION = (
    L15_ROOT /
    "models/l15_final_model_selection.json"
)

L15_HASHES = (
    L15_ROOT /
    "hashes/l15_artifact_sha256_manifest.json"
)

L15_PROTOCOL = (
    L15_ROOT /
    "protocol/l15_l16_one_time_test_protocol.json"
)


# --------------------------------------------------------------------------------------------------
# L16 OUTPUT
# --------------------------------------------------------------------------------------------------

L16_ROOT = (
    ROOT /
    "audit/l16_final_test"
)

REPORT_DIR = (
    L16_ROOT /
    "reports"
)

MODEL_DIR = (
    L16_ROOT /
    "models"
)

CACHE_DIR = (
    L16_ROOT /
    "cache"
)

L13_CACHE = (
    CACHE_DIR /
    "l13b_test_embeddings"
)

PLOT_DIR = (
    L16_ROOT /
    "plots"
)

for p in [
    L16_ROOT,
    REPORT_DIR,
    MODEL_DIR,
    CACHE_DIR,
    L13_CACHE,
    PLOT_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 4. HELPERS
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def save_json(
    obj,
    path
):

    with open(
        path,
        "w"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2
        )


def topk_metrics(
    probabilities,
    targets,
    k=5
):

    order = np.argsort(
        -probabilities,
        axis=1
    )

    pred1 = (
        order[:, 0]
    )

    predk = (
        order[:, :k]
    )

    correct1 = (
        pred1 == targets
    )

    correctk = np.array(
        [
            targets[i]
            in predk[i]

            for i in range(
                len(targets)
            )
        ],
        dtype=bool
    )

    return {
        "pred1":
            pred1,

        "predk":
            predk,

        "correct1":
            correct1,

        "correctk":
            correctk,

        "top1":
            float(
                correct1.mean()
                * 100
            ),

        "top5":
            float(
                correctk.mean()
                * 100
            ),

        "correct_top1":
            int(
                correct1.sum()
            ),

        "correct_top5":
            int(
                correctk.sum()
            ),
    }


def sparse_top5_probability(
    probabilities
):

    n = probabilities.shape[0]

    sparse = np.zeros_like(
        probabilities,
        dtype=np.float64
    )

    top5 = np.argsort(
        -probabilities,
        axis=1
    )[:, :5]

    for i in range(n):

        sparse[
            i,
            top5[i]
        ] = probabilities[
            i,
            top5[i]
        ]

    return sparse


# ==================================================================================================
# 5. START
# ==================================================================================================

print("=" * 140)
print("CISLR — L16")
print("ONE-TIME FINAL 211-CLASS TEST EVALUATION")
print("=" * 140)

print()
print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("Device      :", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU         :",
        torch.cuda.get_device_name(0)
    )

print()
print("FROZEN FINAL SYSTEM")
print("-" * 140)
print("L12B SSL ST-GCN               : 0.40")
print("L13B RGB Temporal Transformer : 0.30")
print("L14A VideoMAE                  : 0.30")
print("Fusion                         : sparse Top-5 raw probability")


# ==================================================================================================
# 6. VERIFY L15 FREEZE
# ==================================================================================================

print("\n" + "=" * 140)
print("1. VERIFYING L15 FREEZE")
print("=" * 140)

for name, path in {

    "L15 master freeze":
        L15_MASTER,

    "L15 selection":
        L15_SELECTION,

    "L15 hash manifest":
        L15_HASHES,

    "L15 test protocol":
        L15_PROTOCOL,

    "L12B checkpoint":
        L12B_CKPT,

    "L13A checkpoint":
        L13A_CKPT,

    "L13B checkpoint":
        L13B_CKPT,

    "L14A checkpoint":
        L14A_CKPT,

    "L9 test manifest":
        TEST_MANIFEST,

    "L9 mapping":
        CLASS_MAPPING,

    "L6 raw directory":
        RAW_DIR,

    "RGB video directory":
        VIDEO_DIR,

}.items():

    ok = path.exists()

    print(
        f"{name:28s}: "
        f"{'PASS' if ok else 'FAIL'}"
    )

    if not ok:
        raise FileNotFoundError(
            path
        )


with open(
    L15_MASTER,
    "r"
) as f:

    master_freeze = json.load(
        f
    )


if (
    master_freeze.get(
        "status"
    )
    !=
    "COMPLETE_AND_FROZEN"
):

    raise RuntimeError(
        "L15 is not frozen."
    )


if (
    master_freeze.get(
        "selected_system"
    )
    !=
    "L14C"
):

    raise RuntimeError(
        "L15 selected system is not L14C."
    )


with open(
    L15_SELECTION,
    "r"
) as f:

    selection = json.load(
        f
    )


frozen_weights = selection[
    "fusion_weights"
]

assert abs(
    float(
        frozen_weights["L12B"]
    )
    -
    0.40
) < 1e-9

assert abs(
    float(
        frozen_weights["L13B"]
    )
    -
    0.30
) < 1e-9

assert abs(
    float(
        frozen_weights["L14A"]
    )
    -
    0.30
) < 1e-9


print()
print(
    "[PASS] L15 model selection is frozen."
)


# ==================================================================================================
# 7. VERIFY FROZEN CHECKPOINT HASHES
# ==================================================================================================

print("\n" + "=" * 140)
print("2. CHECKPOINT SHA256 VERIFICATION")
print("=" * 140)

with open(
    L15_HASHES,
    "r"
) as f:

    frozen_hashes = json.load(
        f
    )


hash_checks = {

    "l12b_checkpoint":
        L12B_CKPT,

    "l13b_checkpoint":
        L13B_CKPT,

    "l14a_checkpoint":
        L14A_CKPT,

}


for key, path in (
    hash_checks.items()
):

    expected = (
        frozen_hashes[
            key
        ][
            "sha256"
        ]
    )

    actual = sha256_file(
        path
    )

    ok = (
        actual == expected
    )

    print(
        f"{key:25s}: "
        f"{'PASS' if ok else 'FAIL'}"
    )

    if not ok:

        raise RuntimeError(
            f"Checkpoint changed after L15: {key}"
        )


print()
print(
    "[PASS] All frozen model checkpoints match L15."
)


# ==================================================================================================
# 8. OPEN THE FROZEN TEST MANIFEST
#
# THIS IS THE POINT WHERE THE FINAL TEST SET IS CONSUMED.
# ==================================================================================================

print("\n" + "=" * 140)
print("3. OPENING FROZEN L9 TEST PARTITION — FINAL EVALUATION")
print("=" * 140)

test_df = pd.read_csv(
    TEST_MANIFEST
)

print(
    "Test rows:",
    len(
        test_df
    )
)

print(
    "Columns:",
    list(
        test_df.columns
    )
)


if len(
    test_df
) != 211:

    raise RuntimeError(
        f"Expected 211 test samples, got {len(test_df)}."
    )


# ==================================================================================================
# 9. CLASS MAPPING
# ==================================================================================================

with open(
    CLASS_MAPPING,
    "r"
) as f:

    mapping_json = json.load(
        f
    )


if (
    isinstance(
        mapping_json,
        dict
    )
    and
    "gloss_to_id"
    in mapping_json
):

    gloss_to_id = {
        str(k):
            int(v)

        for k, v
        in mapping_json[
            "gloss_to_id"
        ].items()
    }

else:

    raise RuntimeError(
        "Expected gloss_to_id in frozen L9 mapping."
    )


id_to_gloss = {
    int(v):
        str(k)

    for k, v
    in gloss_to_id.items()
}


if len(
    gloss_to_id
) != NUM_CLASSES:

    raise RuntimeError(
        "Frozen mapping is not 211 classes."
    )


if set(
    id_to_gloss.keys()
) != set(
    range(
        NUM_CLASSES
    )
):

    raise RuntimeError(
        "Frozen class IDs are not 0..210."
    )


# Determine manifest columns robustly.

uid_col = (
    "uid"
    if "uid" in test_df.columns
    else "uid_clean"
)

gloss_col = (
    "gloss"
    if "gloss" in test_df.columns
    else None
)

class_col = None

for candidate in [
    "class_id",
    "label",
    "target",
    "target_class_id",
]:

    if candidate in test_df.columns:

        class_col = candidate
        break


test_df[
    uid_col
] = (
    test_df[
        uid_col
    ]
    .astype(str)
)


if class_col is not None:

    test_targets = (
        test_df[
            class_col
        ]
        .astype(int)
        .to_numpy()
    )

elif gloss_col is not None:

    test_targets = np.array(
        [
            gloss_to_id[
                str(g)
            ]

            for g in test_df[
                gloss_col
            ]
        ],
        dtype=np.int64
    )

else:

    raise RuntimeError(
        "Could not determine test target column."
    )


if gloss_col is None:

    test_glosses = np.array(
        [
            id_to_gloss[
                int(y)
            ]

            for y in test_targets
        ],
        dtype=object
    )

else:

    test_glosses = (
        test_df[
            gloss_col
        ]
        .astype(str)
        .to_numpy()
    )


test_uids = (
    test_df[
        uid_col
    ]
    .astype(str)
    .tolist()
)


if len(
    set(
        test_uids
    )
) != 211:

    raise RuntimeError(
        "Duplicate test UID detected."
    )


if set(
    test_targets.tolist()
) != set(
    range(
        211
    )
):

    raise RuntimeError(
        "Expected exactly one test sample for every class."
    )


print()
print("[PASS] 211 frozen test samples loaded.")
print("[PASS] Exactly one test sample/class.")
print("[PASS] Frozen class mapping preserved.")


# ==================================================================================================
# 10. INDEX L6 RAW SEQUENCES
# ==================================================================================================

print("\n" + "=" * 140)
print("4. INDEXING RAW LANDMARK SEQUENCES")
print("=" * 140)

raw_files = sorted(
    RAW_DIR.glob(
        "*.npz"
    )
)

uid_to_raw = {}


for i, path in enumerate(
    raw_files,
    start=1
):

    try:

        with np.load(
            path,
            allow_pickle=True
        ) as d:

            uid = str(
                d[
                    "uid"
                ].item()
            )

        uid_to_raw[
            uid
        ] = path

    except Exception:

        pass


    if (
        i % 1000 == 0
        or
        i == len(raw_files)
    ):

        print(
            f"  {i}/{len(raw_files)}"
        )


missing_raw = [
    uid
    for uid in test_uids
    if uid not in uid_to_raw
]


if missing_raw:

    raise RuntimeError(
        "Missing test raw sequences: "
        +
        str(
            missing_raw[:10]
        )
    )


print()
print(
    "[PASS] All 211 test raw sequences located."
)


# ==================================================================================================
# 11. TEST VIDEO PATHS
# ==================================================================================================

uid_to_video = {}


for uid in test_uids:

    candidates = [

        VIDEO_DIR /
        f"{uid}.mp4",

        VIDEO_DIR /
        uid,

    ]

    found = None

    for candidate in candidates:

        if candidate.exists():

            found = candidate
            break


    if found is None:

        matches = list(
            VIDEO_DIR.glob(
                f"{uid}.*"
            )
        )

        if matches:
            found = matches[0]


    if found is None:

        raise FileNotFoundError(
            f"Video missing for test UID: {uid}"
        )


    uid_to_video[
        uid
    ] = found


print(
    "[PASS] All 211 final-test RGB videos located."
)


# ==================================================================================================
# 12. L12B PREPROCESSING
# ==================================================================================================

def uniform_indices(
    length,
    target_length=32
):

    if length <= 0:

        return np.zeros(
            target_length,
            dtype=np.int64
        )

    idx = np.linspace(
        0,
        length - 1,
        target_length
    )

    idx = np.round(
        idx
    ).astype(
        np.int64
    )

    return np.clip(
        idx,
        0,
        length - 1
    )


def interpolate_short_gaps(
    data,
    mask,
    max_gap=3
):

    data = data.copy()

    mask = (
        mask
        .copy()
        .astype(
            np.uint8
        )
    )

    T, H = (
        mask.shape
    )


    for h in range(H):

        valid = np.where(
            mask[
                :,
                h
            ] > 0
        )[0]


        if len(
            valid
        ) < 2:

            continue


        for a, b in zip(
            valid[:-1],
            valid[1:]
        ):

            gap = (
                b -
                a -
                1
            )


            if (
                gap <= 0
                or
                gap > max_gap
            ):

                continue


            for k in range(
                1,
                gap + 1
            ):

                alpha = (
                    k /
                    (
                        gap + 1
                    )
                )

                data[
                    a + k,
                    h
                ] = (

                    (
                        1.0 -
                        alpha
                    )
                    *
                    data[
                        a,
                        h
                    ]

                    +

                    alpha
                    *
                    data[
                        b,
                        h
                    ]
                )

                mask[
                    a + k,
                    h
                ] = 1


    return (
        data,
        mask
    )


def raw_to_graph(
    npz_path
):

    with np.load(
        npz_path,
        allow_pickle=True
    ) as d:

        uid = str(
            d[
                "uid"
            ].item()
        )

        landmarks_px = (
            d[
                "landmarks_px"
            ]
            .astype(
                np.float32
            )
        )

        world = (
            d[
                "world_landmarks"
            ]
            .astype(
                np.float32
            )
        )

        hand_mask = (
            d[
                "hand_mask"
            ]
            .astype(
                np.uint8
            )
        )

        width = float(
            d[
                "width"
            ].item()
        )

        height = float(
            d[
                "height"
            ].item()
        )


    if landmarks_px.ndim != 4:

        raise ValueError(
            f"Bad landmarks shape: {landmarks_px.shape}"
        )


    if world.shape != landmarks_px.shape:

        raise ValueError(
            f"World/image mismatch for {uid}"
        )


    T = (
        landmarks_px.shape[
            0
        ]
    )


    if hand_mask.shape != (
        T,
        2
    ):

        raise ValueError(
            f"Bad hand mask shape for {uid}"
        )


    world, repaired_mask = (
        interpolate_short_gaps(
            world,
            hand_mask,
            max_gap=3
        )
    )


    image = (
        landmarks_px.copy()
    )


    image, _ = (
        interpolate_short_gaps(
            image,
            hand_mask,
            max_gap=3
        )
    )


    world_norm = np.zeros_like(
        world,
        dtype=np.float32
    )


    MCP = [
        5,
        9,
        13,
        17
    ]


    for t in range(T):

        for h in range(2):

            if (
                repaired_mask[
                    t,
                    h
                ]
                ==
                0
            ):

                continue


            pts = (
                world[
                    t,
                    h
                ]
                .copy()
            )

            wrist = (
                pts[
                    0
                ]
                .copy()
            )

            centered = (
                pts -
                wrist
            )

            scale = np.mean(
                np.linalg.norm(
                    centered[
                        MCP
                    ],
                    axis=1
                )
            )


            if (
                not np.isfinite(
                    scale
                )
                or
                scale < 1e-5
            ):

                scale = 1.0


            world_norm[
                t,
                h
            ] = (
                centered /
                scale
            )


    image_xy = np.zeros(
        (
            T,
            2,
            21,
            2
        ),
        dtype=np.float32
    )


    for t in range(T):

        for h in range(2):

            if (
                repaired_mask[
                    t,
                    h
                ]
                ==
                0
            ):

                continue


            image_xy[
                t,
                h,
                :,
                0
            ] = (

                image[
                    t,
                    h,
                    :,
                    0
                ]
                /
                max(
                    width,
                    1.0
                )

                -
                0.5
            )


            image_xy[
                t,
                h,
                :,
                1
            ] = (

                image[
                    t,
                    h,
                    :,
                    1
                ]
                /
                max(
                    height,
                    1.0
                )

                -
                0.5
            )


    world_vel = np.zeros_like(
        world_norm,
        dtype=np.float32
    )

    image_vel = np.zeros_like(
        image_xy,
        dtype=np.float32
    )


    for t in range(
        1,
        T
    ):

        for h in range(2):

            if (
                repaired_mask[
                    t,
                    h
                ] > 0
                and
                repaired_mask[
                    t - 1,
                    h
                ] > 0
            ):

                world_vel[
                    t,
                    h
                ] = (

                    world_norm[
                        t,
                        h
                    ]
                    -
                    world_norm[
                        t - 1,
                        h
                    ]
                )


                image_vel[
                    t,
                    h
                ] = (

                    image_xy[
                        t,
                        h
                    ]
                    -
                    image_xy[
                        t - 1,
                        h
                    ]
                )


    idx = uniform_indices(
        T,
        SEQ_LEN
    )


    world_norm = (
        world_norm[
            idx
        ]
    )

    world_vel = (
        world_vel[
            idx
        ]
    )

    image_xy = (
        image_xy[
            idx
        ]
    )

    image_vel = (
        image_vel[
            idx
        ]
    )

    sampled_mask = (
        repaired_mask[
            idx
        ]
    )


    world_norm = (
        world_norm
        .reshape(
            SEQ_LEN,
            NUM_NODES,
            3
        )
    )

    world_vel = (
        world_vel
        .reshape(
            SEQ_LEN,
            NUM_NODES,
            3
        )
    )

    image_xy = (
        image_xy
        .reshape(
            SEQ_LEN,
            NUM_NODES,
            2
        )
    )

    image_vel = (
        image_vel
        .reshape(
            SEQ_LEN,
            NUM_NODES,
            2
        )
    )


    node_mask = np.repeat(
        sampled_mask[
            :,
            :,
            None
        ],
        JOINTS_PER_HAND,
        axis=2
    ).reshape(
        SEQ_LEN,
        NUM_NODES
    ).astype(
        np.float32
    )


    m = (
        node_mask[
            ...,
            None
        ]
    )


    world_norm *= m
    world_vel *= m
    image_xy *= m
    image_vel *= m


    graph = np.concatenate(
        [
            world_norm,
            world_vel,
            image_xy,
            image_vel,
            node_mask[
                ...,
                None
            ],
        ],
        axis=-1
    )


    graph = np.transpose(
        graph,
        (
            2,
            0,
            1
        )
    ).astype(
        np.float32
    )


    if graph.shape != (
        11,
        32,
        42
    ):

        raise RuntimeError(
            f"Bad graph shape {uid}: {graph.shape}"
        )


    if not np.isfinite(
        graph
    ).all():

        raise RuntimeError(
            f"Non-finite graph: {uid}"
        )


    return graph


# ==================================================================================================
# 13. L12B MODEL
# ==================================================================================================

class GraphConv(
    nn.Module
):

    def __init__(
        self,
        in_channels,
        out_channels,
        num_partitions=3
    ):

        super().__init__()

        self.num_partitions = (
            num_partitions
        )

        self.out_channels = (
            out_channels
        )

        self.conv = nn.Conv2d(
            in_channels,
            out_channels
            *
            num_partitions,
            kernel_size=1,
            bias=False
        )


    def forward(
        self,
        x,
        A
    ):

        x = self.conv(
            x
        )

        N, KC, T, V = (
            x.shape
        )

        x = x.view(
            N,
            self.num_partitions,
            self.out_channels,
            T,
            V
        )

        x = torch.einsum(
            "nkctv,kvw->nctw",
            x,
            A
        )

        return (
            x.contiguous()
        )


class STGCNBlock(
    nn.Module
):

    def __init__(
        self,
        in_channels,
        out_channels,
        dropout=0.35
    ):

        super().__init__()


        self.gcn = GraphConv(
            in_channels,
            out_channels,
            num_partitions=3
        )


        self.gcn_bn = (
            nn.BatchNorm2d(
                out_channels
            )
        )


        self.temporal = (
            nn.Sequential(

                nn.ReLU(
                    inplace=True
                ),

                nn.Conv2d(
                    out_channels,
                    out_channels,
                    kernel_size=(
                        9,
                        1
                    ),
                    padding=(
                        4,
                        0
                    ),
                    bias=False
                ),

                nn.BatchNorm2d(
                    out_channels
                ),

                nn.Dropout(
                    dropout
                )
            )
        )


        if (
            in_channels
            ==
            out_channels
        ):

            self.residual = (
                nn.Identity()
            )

        else:

            self.residual = (
                nn.Sequential(

                    nn.Conv2d(
                        in_channels,
                        out_channels,
                        kernel_size=1,
                        bias=False
                    ),

                    nn.BatchNorm2d(
                        out_channels
                    )
                )
            )


        self.relu = nn.ReLU(
            inplace=True
        )


    def forward(
        self,
        x,
        A
    ):

        residual = (
            self.residual(
                x
            )
        )

        x = self.gcn(
            x,
            A
        )

        x = self.gcn_bn(
            x
        )

        x = self.temporal(
            x
        )

        x = (
            x +
            residual
        )

        return self.relu(
            x
        )


class STGCNEncoder(
    nn.Module
):

    def __init__(
        self,
        in_channels=11,
        dropout=0.35
    ):

        super().__init__()


        self.input_bn = (
            nn.BatchNorm1d(
                in_channels
                *
                NUM_NODES
            )
        )


        self.block1 = STGCNBlock(
            in_channels,
            64,
            dropout=
                dropout
                *
                0.5
        )

        self.block2 = STGCNBlock(
            64,
            64,
            dropout=
                dropout
        )

        self.block3 = STGCNBlock(
            64,
            128,
            dropout=
                dropout
        )

        self.block4 = STGCNBlock(
            128,
            128,
            dropout=
                dropout
        )

        self.output_norm = (
            nn.LayerNorm(
                128
            )
        )


    def forward(
        self,
        x,
        A
    ):

        N, C, T, V = (
            x.shape
        )


        y = x.permute(
            0,
            3,
            1,
            2
        ).contiguous()


        y = y.view(
            N,
            V * C,
            T
        )


        y = self.input_bn(
            y
        )


        y = y.view(
            N,
            V,
            C,
            T
        )


        x = y.permute(
            0,
            2,
            3,
            1
        ).contiguous()


        x = self.block1(
            x,
            A
        )

        x = self.block2(
            x,
            A
        )

        x = self.block3(
            x,
            A
        )

        x = self.block4(
            x,
            A
        )


        mask = (
            y.view(
                N,
                V,
                C,
                T
            )
            [
                :,
                :,
                10,
                :
            ]
        )


        mask = (
            mask
            .permute(
                0,
                2,
                1
            )
            .unsqueeze(
                1
            )
        )


        mask = (
            mask > 0.5
        ).float()


        weighted = (
            x *
            mask
        )


        denom = (
            mask
            .sum(
                dim=(
                    2,
                    3
                )
            )
            .clamp(
                min=1.0
            )
        )


        pooled = (
            weighted
            .sum(
                dim=(
                    2,
                    3
                )
            )
            /
            denom
        )


        pooled = (
            self.output_norm(
                pooled
            )
        )


        return pooled


class STGCNClassifier(
    nn.Module
):

    def __init__(
        self,
        num_classes=211
    ):

        super().__init__()


        self.encoder = STGCNEncoder(
            in_channels=11,
            dropout=L12_DROPOUT
        )


        self.dropout = (
            nn.Dropout(
                L12_DROPOUT
            )
        )


        self.classifier = (
            nn.Linear(
                128,
                num_classes
            )
        )


    def forward(
        self,
        x,
        A
    ):

        features = (
            self.encoder(
                x,
                A
            )
        )

        features = (
            self.dropout(
                features
            )
        )

        return (
            self.classifier(
                features
            )
        )


# ==================================================================================================
# 14. LOAD L12B
# ==================================================================================================

print("\n" + "=" * 140)
print("5. LOADING FROZEN L12B")
print("=" * 140)

l12_ckpt = torch.load(
    L12B_CKPT,
    map_location="cpu",
    weights_only=False
)


channel_mean = np.asarray(
    l12_ckpt[
        "channel_mean"
    ],
    dtype=np.float32
)

channel_std = np.asarray(
    l12_ckpt[
        "channel_std"
    ],
    dtype=np.float32
)

A_l12 = torch.from_numpy(
    np.asarray(
        l12_ckpt[
            "adjacency"
        ],
        dtype=np.float32
    )
).to(
    DEVICE
)


l12_model = (
    STGCNClassifier(
        NUM_CLASSES
    )
    .to(
        DEVICE
    )
)


load_result = (
    l12_model
    .load_state_dict(
        l12_ckpt[
            "model_state_dict"
        ],
        strict=True
    )
)


l12_model.eval()


print(
    "Missing keys    :",
    load_result.missing_keys
)

print(
    "Unexpected keys :",
    load_result.unexpected_keys
)

print(
    "[PASS] L12B loaded exactly."
)


def normalize_l12(
    graph
):

    graph = (
        graph.copy()
    )

    mask = (
        graph[
            10:11
        ]
        .copy()
    )


    for c in range(10):

        graph[
            c:c+1
        ] = (

            graph[
                c:c+1
            ]

            -

            channel_mean[
                c
            ]

        ) / max(
            float(
                channel_std[
                    c
                ]
            ),
            1e-6
        )


        graph[
            c:c+1
        ] *= mask


    graph[
        10:11
    ] = mask


    return (
        graph
        .astype(
            np.float32
        )
    )


# ==================================================================================================
# 15. L12B TEST INFERENCE
# ==================================================================================================

print("\n" + "=" * 140)
print("6. L12B FINAL TEST INFERENCE")
print("=" * 140)

l12_probs = []

l12_times = []


with torch.inference_mode():

    for i, uid in enumerate(
        test_uids,
        start=1
    ):

        graph = raw_to_graph(
            uid_to_raw[
                uid
            ]
        )

        graph = normalize_l12(
            graph
        )


        x = (
            torch
            .from_numpy(
                graph
            )
            .unsqueeze(
                0
            )
            .to(
                DEVICE
            )
        )


        if DEVICE.type == "cuda":
            torch.cuda.synchronize()


        t0 = time.perf_counter()


        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):

            logits = (
                l12_model(
                    x,
                    A_l12
                )
            )


        if DEVICE.type == "cuda":
            torch.cuda.synchronize()


        elapsed = (
            time.perf_counter()
            -
            t0
        )


        prob = (
            torch.softmax(
                logits,
                dim=1
            )
            .float()
            .cpu()
            .numpy()[0]
        )


        l12_probs.append(
            prob
        )

        l12_times.append(
            elapsed
        )


        if (
            i % 25 == 0
            or
            i == 211
        ):

            print(
                f"  {i:3d}/211"
            )


l12_probs = np.asarray(
    l12_probs,
    dtype=np.float64
)

l12_metrics = topk_metrics(
    l12_probs,
    test_targets
)


print()
print(
    f"L12B TEST Top-1 : "
    f"{l12_metrics['top1']:.6f}% "
    f"({l12_metrics['correct_top1']}/211)"
)

print(
    f"L12B TEST Top-5 : "
    f"{l12_metrics['top5']:.6f}% "
    f"({l12_metrics['correct_top5']}/211)"
)


# Free GPU before RGB branches.

del l12_model

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


# ==================================================================================================
# 16. L13A VISUAL ENCODER
# ==================================================================================================

class VisualEncoder(
    nn.Module
):

    def __init__(
        self,
        backbone
    ):

        super().__init__()

        self.features = (
            backbone.features
        )

        self.avgpool = (
            nn.AdaptiveAvgPool2d(
                1
            )
        )


    def forward(
        self,
        x
    ):

        x = self.features(
            x
        )

        x = self.avgpool(
            x
        )

        x = torch.flatten(
            x,
            1
        )

        return x


# ==================================================================================================
# 17. LOAD L13A
# ==================================================================================================

print("\n" + "=" * 140)
print("7. LOADING FROZEN L13A VISUAL ENCODER")
print("=" * 140)

backbone = (
    mobilenet_v3_small(
        weights=None
    )
)

visual_encoder = (
    VisualEncoder(
        backbone
    )
)


l13a_ckpt = torch.load(
    L13A_CKPT,
    map_location="cpu",
    weights_only=False
)


encoder_state = None


if isinstance(
    l13a_ckpt,
    dict
):

    for key in [

        "encoder_state_dict",
        "visual_encoder_state_dict",
        "backbone_state_dict",
        "model_state_dict",
        "state_dict",

    ]:

        if (
            key
            in
            l13a_ckpt
            and
            isinstance(
                l13a_ckpt[
                    key
                ],
                dict
            )
        ):

            encoder_state = (
                l13a_ckpt[
                    key
                ]
            )

            break


if encoder_state is None:

    if (
        isinstance(
            l13a_ckpt,
            dict
        )
        and
        all(
            torch.is_tensor(
                v
            )
            for v
            in l13a_ckpt.values()
        )
    ):

        encoder_state = (
            l13a_ckpt
        )


if encoder_state is None:

    raise RuntimeError(
        "Could not locate L13A encoder state."
    )


clean_state = {}


for key, value in (
    encoder_state.items()
):

    new_key = key


    for prefix in [

        "module.",
        "encoder.",
        "visual_encoder.",

    ]:

        if new_key.startswith(
            prefix
        ):

            new_key = new_key[
                len(
                    prefix
                ):
            ]


    clean_state[
        new_key
    ] = value


try:

    visual_encoder.load_state_dict(
        clean_state,
        strict=True
    )

except RuntimeError:

    expected_keys = set(
        visual_encoder
        .state_dict()
        .keys()
    )

    filtered_state = {

        k: v

        for k, v
        in clean_state.items()

        if k in expected_keys
    }


    missing = (
        expected_keys
        -
        set(
            filtered_state.keys()
        )
    )


    if missing:

        raise RuntimeError(
            "Could not reconstruct exact L13A encoder. "
            f"Missing {list(sorted(missing))[:10]}"
        )


    visual_encoder.load_state_dict(
        filtered_state,
        strict=True
    )


visual_encoder = (
    visual_encoder
    .to(
        DEVICE
    )
)

visual_encoder.eval()


for p in (
    visual_encoder.parameters()
):

    p.requires_grad = False


print(
    "[PASS] L13A visual encoder loaded."
)


# ==================================================================================================
# 18. L13B IMAGE PREPROCESSING
# ==================================================================================================

visual_transform = (
    transforms.Compose(
        [

            transforms.ToPILImage(),

            transforms.Resize(
                (
                    IMAGE_SIZE,
                    IMAGE_SIZE
                )
            ),

            transforms.ToTensor(),

            transforms.Normalize(
                mean=[
                    0.485,
                    0.456,
                    0.406
                ],

                std=[
                    0.229,
                    0.224,
                    0.225
                ]
            )
        ]
    )
)


def clip_box(
    box,
    width,
    height
):

    x1, y1, x2, y2 = (
        box
    )


    x1 = int(
        np.clip(
            round(
                x1
            ),
            0,
            max(
                width - 1,
                0
            )
        )
    )


    y1 = int(
        np.clip(
            round(
                y1
            ),
            0,
            max(
                height - 1,
                0
            )
        )
    )


    x2 = int(
        np.clip(
            round(
                x2
            ),
            x1 + 1,
            width
        )
    )


    y2 = int(
        np.clip(
            round(
                y2
            ),
            y1 + 1,
            height
        )
    )


    return (
        x1,
        y1,
        x2,
        y2
    )


def square_box_from_points(
    points_xy,
    width,
    height,
    scale,
    minimum_size
):

    points_xy = np.asarray(
        points_xy,
        dtype=np.float32
    )


    finite = np.isfinite(
        points_xy
    ).all(
        axis=1
    )


    points_xy = (
        points_xy[
            finite
        ]
    )


    if len(
        points_xy
    ) == 0:

        return None


    xmin = float(
        points_xy[
            :,
            0
        ].min()
    )

    xmax = float(
        points_xy[
            :,
            0
        ].max()
    )

    ymin = float(
        points_xy[
            :,
            1
        ].min()
    )

    ymax = float(
        points_xy[
            :,
            1
        ].max()
    )


    cx = (
        xmin +
        xmax
    ) / 2.0

    cy = (
        ymin +
        ymax
    ) / 2.0


    side = max(

        xmax - xmin,
        ymax - ymin,

        float(
            minimum_size
        )
    )


    side *= scale


    return clip_box(
        (
            cx -
            side / 2.0,

            cy -
            side / 2.0,

            cx +
            side / 2.0,

            cy +
            side / 2.0
        ),
        width,
        height
    )


def context_box_from_landmarks(
    landmarks,
    mask,
    width,
    height
):

    all_points = []


    for hand_idx in range(
        2
    ):

        if (
            mask[
                hand_idx
            ]
            <=
            0
        ):

            continue


        points = (
            landmarks[
                hand_idx,
                :,
                :2
            ]
        )


        finite = np.isfinite(
            points
        ).all(
            axis=1
        )


        points = (
            points[
                finite
            ]
        )


        if len(
            points
        ) > 0:

            all_points.append(
                points
            )


    if len(
        all_points
    ) == 0:

        return None


    all_points = (
        np.concatenate(
            all_points,
            axis=0
        )
    )


    return square_box_from_points(
        all_points,
        width,
        height,
        CONTEXT_BOX_SCALE,
        MIN_CONTEXT_BOX
    )


def resample_positions(
    T,
    target_length=32
):

    if T <= 0:

        raise RuntimeError(
            "Empty raw sequence."
        )


    indices = np.linspace(
        0,
        T - 1,
        target_length
    )


    indices = np.rint(
        indices
    ).astype(
        np.int64
    )


    return np.clip(
        indices,
        0,
        T - 1
    )


def decode_selected_frames(
    video_path,
    frame_numbers
):

    cap = cv2.VideoCapture(
        str(
            video_path
        )
    )


    if not cap.isOpened():

        cap.release()

        raise RuntimeError(
            f"Could not open {video_path}"
        )


    frames = []

    last_good = None


    for frame_number in (
        frame_numbers
    ):

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(
                frame_number
            )
        )


        ok, frame = (
            cap.read()
        )


        if (
            not ok
            or
            frame is None
        ):

            recovered = None


            for offset in [

                -1,
                1,
                -2,
                2,
                -3,
                3,

            ]:

                candidate = max(
                    0,
                    int(
                        frame_number
                    )
                    +
                    offset
                )


                cap.set(
                    cv2.CAP_PROP_POS_FRAMES,
                    candidate
                )


                ok2, frame2 = (
                    cap.read()
                )


                if (
                    ok2
                    and
                    frame2
                    is not None
                ):

                    recovered = (
                        frame2
                    )

                    break


            if recovered is not None:

                frame = (
                    recovered
                )

            elif last_good is not None:

                frame = (
                    last_good
                    .copy()
                )

            else:

                cap.release()

                raise RuntimeError(
                    f"Cannot decode {frame_number} from {video_path}"
                )


        last_good = (
            frame.copy()
        )


        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )


        frames.append(
            frame
        )


    cap.release()

    return frames


def deterministic_visual_crop(
    frame,
    landmarks,
    mask
):

    height, width = (
        frame.shape[
            :2
        ]
    )


    box = (
        context_box_from_landmarks(
            landmarks,
            mask,
            width,
            height
        )
    )


    if box is None:

        return (
            frame,
            0
        )


    x1, y1, x2, y2 = (
        box
    )


    crop = (
        frame[
            y1:y2,
            x1:x2
        ]
    )


    if (
        crop.size == 0
        or
        crop.shape[0] < 8
        or
        crop.shape[1] < 8
    ):

        return (
            frame,
            0
        )


    return (
        crop,
        1
    )


# ==================================================================================================
# 19. EXTRACT L13B TEST EMBEDDING
# ==================================================================================================

@torch.no_grad()
def extract_l13_embedding(
    uid
):

    raw_path = (
        uid_to_raw[
            uid
        ]
    )

    video_path = (
        uid_to_video[
            uid
        ]
    )


    with np.load(
        raw_path,
        allow_pickle=True
    ) as d:

        frame_numbers = (
            d[
                "frame_numbers"
            ]
            .astype(
                np.int64
            )
        )

        landmarks = (
            d[
                "landmarks_px"
            ]
            .astype(
                np.float32
            )
        )

        hand_mask = (
            d[
                "hand_mask"
            ]
            .astype(
                np.uint8
            )
        )


    T = len(
        frame_numbers
    )


    positions = (
        resample_positions(
            T,
            32
        )
    )


    selected_frame_numbers = (
        frame_numbers[
            positions
        ]
    )

    selected_landmarks = (
        landmarks[
            positions
        ]
    )

    selected_masks = (
        hand_mask[
            positions
        ]
    )


    frames = (
        decode_selected_frames(
            video_path,
            selected_frame_numbers
        )
    )


    tensors = []


    for i in range(
        32
    ):

        crop, _ = (
            deterministic_visual_crop(
                frames[i],
                selected_landmarks[i],
                selected_masks[i]
            )
        )


        tensors.append(
            visual_transform(
                crop
            )
        )


    tensors = torch.stack(
        tensors,
        dim=0
    )


    embeddings = []


    for start in range(
        0,
        32,
        EMBED_BATCH_SIZE
    ):

        batch = (
            tensors[
                start:
                start
                +
                EMBED_BATCH_SIZE
            ]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):

            emb = (
                visual_encoder(
                    batch
                )
            )


        embeddings.append(
            emb
            .float()
            .cpu()
        )


    embeddings = (
        torch.cat(
            embeddings,
            dim=0
        )
        .numpy()
        .astype(
            np.float32
        )
    )


    if embeddings.shape != (
        32,
        576
    ):

        raise RuntimeError(
            f"Bad L13 embedding {uid}: {embeddings.shape}"
        )


    return embeddings


# ==================================================================================================
# 20. BUILD TEST EMBEDDING CACHE
#
# This is still the SAME single L16 evaluation.
# Cache is only for crash recovery / reproducibility.
# ==================================================================================================

print("\n" + "=" * 140)
print("8. EXTRACTING L13B FINAL TEST RGB EMBEDDINGS")
print("=" * 140)

l13_embedding_sequences = []


for i, uid in enumerate(
    test_uids,
    start=1
):

    cache_path = (
        L13_CACHE /
        f"{uid}.npz"
    )


    if cache_path.exists():

        with np.load(
            cache_path
        ) as d:

            embedding = (
                d[
                    "embeddings"
                ]
                .astype(
                    np.float32
                )
            )


        if embedding.shape != (
            32,
            576
        ):

            raise RuntimeError(
                f"Invalid existing cache for {uid}"
            )


    else:

        embedding = (
            extract_l13_embedding(
                uid
            )
        )


        np.savez_compressed(
            cache_path,
            uid=uid,
            embeddings=embedding
        )


    l13_embedding_sequences.append(
        embedding
    )


    if (
        i % 25 == 0
        or
        i == 211
    ):

        print(
            f"  {i:3d}/211"
        )


X_l13 = np.stack(
    l13_embedding_sequences,
    axis=0
).astype(
    np.float32
)


print(
    "Test embedding shape:",
    X_l13.shape
)

assert X_l13.shape == (
    211,
    32,
    576
)


# Free L13A.

del visual_encoder
del backbone

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


# ==================================================================================================
# 21. L13B TEMPORAL MODEL
# ==================================================================================================

class LearnablePositionalEncoding(
    nn.Module
):

    def __init__(
        self,
        sequence_length,
        model_dim
    ):

        super().__init__()


        self.position = (
            nn.Parameter(
                torch.zeros(
                    1,
                    sequence_length,
                    model_dim
                )
            )
        )


        nn.init.trunc_normal_(
            self.position,
            std=0.02
        )


    def forward(
        self,
        x
    ):

        return (
            x
            +
            self.position[
                :,
                :x.size(1)
            ]
        )


class RGBTemporalTransformer(
    nn.Module
):

    def __init__(
        self,
        input_dim=576,
        model_dim=256,
        num_heads=4,
        num_layers=2,
        ff_dim=512,
        num_classes=211,
        dropout=0.35,
        sequence_length=32
    ):

        super().__init__()


        self.input_norm = (
            nn.LayerNorm(
                input_dim
            )
        )


        self.input_projection = (
            nn.Linear(
                input_dim,
                model_dim
            )
        )


        self.position = (
            LearnablePositionalEncoding(
                sequence_length,
                model_dim
            )
        )


        encoder_layer = (
            nn.TransformerEncoderLayer(

                d_model=
                    model_dim,

                nhead=
                    num_heads,

                dim_feedforward=
                    ff_dim,

                dropout=
                    dropout,

                activation=
                    "gelu",

                batch_first=
                    True,

                norm_first=
                    True
            )
        )


        self.transformer = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=
                    num_layers
            )
        )


        self.temporal_attention = (
            nn.Sequential(

                nn.Linear(
                    model_dim,
                    128
                ),

                nn.Tanh(),

                nn.Linear(
                    128,
                    1
                )
            )
        )


        self.output_norm = (
            nn.LayerNorm(
                model_dim
            )
        )


        self.dropout = (
            nn.Dropout(
                dropout
            )
        )


        self.classifier = (
            nn.Linear(
                model_dim,
                num_classes
            )
        )


    def forward(
        self,
        x,
        return_attention=False
    ):

        x = self.input_norm(
            x
        )

        x = self.input_projection(
            x
        )

        x = self.position(
            x
        )

        x = self.transformer(
            x
        )


        attention_logits = (
            self.temporal_attention(
                x
            )
            .squeeze(
                -1
            )
        )


        attention = torch.softmax(
            attention_logits,
            dim=1
        )


        pooled = torch.sum(
            x
            *
            attention.unsqueeze(
                -1
            ),
            dim=1
        )


        pooled = self.output_norm(
            pooled
        )

        pooled = self.dropout(
            pooled
        )


        logits = self.classifier(
            pooled
        )


        if return_attention:

            return (
                logits,
                attention
            )


        return logits


# ==================================================================================================
# 22. LOAD L13B
# ==================================================================================================

print("\n" + "=" * 140)
print("9. LOADING FROZEN L13B")
print("=" * 140)

l13_ckpt = torch.load(
    L13B_CKPT,
    map_location="cpu",
    weights_only=False
)


l13_model = (
    RGBTemporalTransformer(

        input_dim=int(
            l13_ckpt[
                "visual_dim"
            ]
        ),

        model_dim=int(
            l13_ckpt[
                "model_dim"
            ]
        ),

        num_heads=int(
            l13_ckpt[
                "num_heads"
            ]
        ),

        num_layers=int(
            l13_ckpt[
                "num_layers"
            ]
        ),

        ff_dim=int(
            l13_ckpt[
                "ff_dim"
            ]
        ),

        num_classes=int(
            l13_ckpt[
                "num_classes"
            ]
        ),

        dropout=float(
            l13_ckpt[
                "dropout"
            ]
        ),

        sequence_length=int(
            l13_ckpt[
                "sequence_length"
            ]
        )

    )
    .to(
        DEVICE
    )
)


result = (
    l13_model
    .load_state_dict(
        l13_ckpt[
            "model_state_dict"
        ],
        strict=True
    )
)


l13_model.eval()


print(
    "Missing keys    :",
    result.missing_keys
)

print(
    "Unexpected keys :",
    result.unexpected_keys
)

print(
    "[PASS] L13B loaded exactly."
)


# ==================================================================================================
# 23. L13B TRAIN-ONLY STANDARDIZATION
# ==================================================================================================

train_mean_l13 = np.asarray(
    l13_ckpt[
        "train_mean"
    ],
    dtype=np.float32
)

train_std_l13 = np.asarray(
    l13_ckpt[
        "train_std"
    ],
    dtype=np.float32
)


train_std_l13 = np.where(
    train_std_l13 < 1e-6,
    1.0,
    train_std_l13
)


X_l13_norm = (

    X_l13
    -
    train_mean_l13[
        None,
        None,
        :
    ]

) / train_std_l13[
    None,
    None,
    :
]


# ==================================================================================================
# 24. L13B TEST INFERENCE
# ==================================================================================================

print("\n" + "=" * 140)
print("10. L13B FINAL TEST INFERENCE")
print("=" * 140)

l13_probs = []

l13_times = []

L13_BATCH = 32


with torch.inference_mode():

    for start in range(
        0,
        211,
        L13_BATCH
    ):

        end = min(
            start +
            L13_BATCH,
            211
        )


        x = torch.from_numpy(
            X_l13_norm[
                start:end
            ]
        ).to(
            DEVICE
        )


        if DEVICE.type == "cuda":
            torch.cuda.synchronize()


        t0 = time.perf_counter()


        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):

            logits = (
                l13_model(
                    x
                )
            )


        if DEVICE.type == "cuda":
            torch.cuda.synchronize()


        elapsed = (
            time.perf_counter()
            -
            t0
        )


        batch_prob = (
            torch.softmax(
                logits,
                dim=1
            )
            .float()
            .cpu()
            .numpy()
        )


        l13_probs.append(
            batch_prob
        )


        per_sample = (
            elapsed /
            (
                end -
                start
            )
        )


        l13_times.extend(
            [
                per_sample
            ]
            *
            (
                end -
                start
            )
        )


        print(
            f"  {end:3d}/211"
        )


l13_probs = np.concatenate(
    l13_probs,
    axis=0
).astype(
    np.float64
)


l13_metrics = topk_metrics(
    l13_probs,
    test_targets
)


print()
print(
    f"L13B TEST Top-1 : "
    f"{l13_metrics['top1']:.6f}% "
    f"({l13_metrics['correct_top1']}/211)"
)

print(
    f"L13B TEST Top-5 : "
    f"{l13_metrics['top5']:.6f}% "
    f"({l13_metrics['correct_top5']}/211)"
)


del l13_model
del X_l13_norm

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


# ==================================================================================================
# 25. L14A VIDEOMAE — LOAD FROZEN CHECKPOINT
# ==================================================================================================

print("\n" + "=" * 140)
print("11. LOADING FROZEN L14A VIDEOMAE")
print("=" * 140)

l14_ckpt = torch.load(
    L14A_CKPT,
    map_location="cpu",
    weights_only=False
)


PRETRAINED_MODEL = str(
    l14_ckpt[
        "pretrained_model"
    ]
)

VIDEOMAE_FRAMES = int(
    l14_ckpt[
        "num_frames"
    ]
)

VIDEOMAE_SIZE = int(
    l14_ckpt[
        "image_size"
    ]
)


print(
    "Base model   :",
    PRETRAINED_MODEL
)

print(
    "Frames       :",
    VIDEOMAE_FRAMES
)

print(
    "Image size   :",
    VIDEOMAE_SIZE
)

print(
    "Classes      :",
    l14_ckpt[
        "num_classes"
    ]
)


# Processor uses the same pretrained VideoMAE image preprocessing.

video_processor = (
    VideoMAEImageProcessor
    .from_pretrained(
        PRETRAINED_MODEL
    )
)


video_model = (
    VideoMAEForVideoClassification
    .from_pretrained(

        PRETRAINED_MODEL,

        num_labels=
            NUM_CLASSES,

        ignore_mismatched_sizes=
            True
    )
)


load_result = (
    video_model
    .load_state_dict(
        l14_ckpt[
            "model_state_dict"
        ],
        strict=True
    )
)


video_model = (
    video_model
    .to(
        DEVICE
    )
)

video_model.eval()


print(
    "Missing keys    :",
    load_result.missing_keys
)

print(
    "Unexpected keys :",
    load_result.unexpected_keys
)

print(
    "[PASS] L14A frozen VideoMAE loaded exactly."
)


# ==================================================================================================
# 26. L14A VIDEO DECODING
# ==================================================================================================

def decode_uniform_video_frames(
    video_path,
    num_frames=16
):

    cap = cv2.VideoCapture(
        str(
            video_path
        )
    )


    if not cap.isOpened():

        cap.release()

        raise RuntimeError(
            f"Cannot open video: {video_path}"
        )


    frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )


    if frame_count <= 0:

        cap.release()

        raise RuntimeError(
            f"Invalid frame count: {video_path}"
        )


    indices = np.linspace(
        0,
        frame_count - 1,
        num_frames
    )


    indices = np.rint(
        indices
    ).astype(
        np.int64
    )


    indices = np.clip(
        indices,
        0,
        frame_count - 1
    )


    frames = []

    last_good = None


    for frame_idx in indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(
                frame_idx
            )
        )


        ok, frame = (
            cap.read()
        )


        if (
            not ok
            or
            frame is None
        ):

            recovered = None


            for offset in [

                -1,
                1,
                -2,
                2,
                -3,
                3,

            ]:

                candidate = int(
                    np.clip(
                        frame_idx
                        +
                        offset,
                        0,
                        frame_count - 1
                    )
                )


                cap.set(
                    cv2.CAP_PROP_POS_FRAMES,
                    candidate
                )


                ok2, frame2 = (
                    cap.read()
                )


                if (
                    ok2
                    and
                    frame2 is not None
                ):

                    recovered = (
                        frame2
                    )

                    break


            if recovered is not None:

                frame = (
                    recovered
                )

            elif last_good is not None:

                frame = (
                    last_good
                    .copy()
                )

            else:

                cap.release()

                raise RuntimeError(
                    f"Could not decode {video_path}"
                )


        last_good = (
            frame.copy()
        )


        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )


        frames.append(
            frame
        )


    cap.release()


    if len(
        frames
    ) != num_frames:

        raise RuntimeError(
            "Incorrect VideoMAE frame count."
        )


    return frames


# ==================================================================================================
# 27. L14A FINAL TEST INFERENCE
# ==================================================================================================

print("\n" + "=" * 140)
print("12. L14A FINAL TEST INFERENCE")
print("=" * 140)

l14_probs = []

l14_times = []


with torch.inference_mode():

    for i, uid in enumerate(
        test_uids,
        start=1
    ):

        frames = (
            decode_uniform_video_frames(
                uid_to_video[
                    uid
                ],
                VIDEOMAE_FRAMES
            )
        )


        inputs = (
            video_processor(
                frames,
                return_tensors="pt"
            )
        )


        pixel_values = (
            inputs[
                "pixel_values"
            ]
            .to(
                DEVICE
            )
        )


        if DEVICE.type == "cuda":
            torch.cuda.synchronize()


        t0 = time.perf_counter()


        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):

            output = (
                video_model(
                    pixel_values=
                        pixel_values
                )
            )


        if DEVICE.type == "cuda":
            torch.cuda.synchronize()


        elapsed = (
            time.perf_counter()
            -
            t0
        )


        prob = (
            torch.softmax(
                output.logits,
                dim=1
            )
            .float()
            .cpu()
            .numpy()[0]
        )


        l14_probs.append(
            prob
        )

        l14_times.append(
            elapsed
        )


        del frames
        del inputs
        del pixel_values
        del output


        if (
            i % 10 == 0
            or
            i == 211
        ):

            print(
                f"  {i:3d}/211"
            )


l14_probs = np.asarray(
    l14_probs,
    dtype=np.float64
)


l14_metrics = topk_metrics(
    l14_probs,
    test_targets
)


print()
print(
    f"L14A TEST Top-1 : "
    f"{l14_metrics['top1']:.6f}% "
    f"({l14_metrics['correct_top1']}/211)"
)

print(
    f"L14A TEST Top-5 : "
    f"{l14_metrics['top5']:.6f}% "
    f"({l14_metrics['correct_top5']}/211)"
)


del video_model

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


# ==================================================================================================
# 28. CONVERT TO EXACT L14C SPARSE TOP-5 REPRESENTATION
# ==================================================================================================

print("\n" + "=" * 140)
print("13. APPLYING FROZEN L14C REPRESENTATION")
print("=" * 140)

S12 = sparse_top5_probability(
    l12_probs
)

S13 = sparse_top5_probability(
    l13_probs
)

S14 = sparse_top5_probability(
    l14_probs
)


assert S12.shape == (
    211,
    211
)

assert S13.shape == (
    211,
    211
)

assert S14.shape == (
    211,
    211
)


print(
    "[PASS] All three branches converted "
    "to sparse Top-5 raw-probability representation."
)


# ==================================================================================================
# 29. FROZEN L14C FUSION
# ==================================================================================================

print("\n" + "=" * 140)
print("14. FROZEN L14C TEST FUSION")
print("=" * 140)

FUSED = (

    W_L12B
    *
    S12

    +

    W_L13B
    *
    S13

    +

    W_L14A
    *
    S14
)


fused_metrics = topk_metrics(
    FUSED,
    test_targets
)


print()
print(
    "Weights:"
)

print(
    "  L12B =",
    W_L12B
)

print(
    "  L13B =",
    W_L13B
)

print(
    "  L14A =",
    W_L14A
)


print()
print(
    f"FINAL TEST TOP-1 : "
    f"{fused_metrics['top1']:.6f}% "
    f"({fused_metrics['correct_top1']}/211)"
)

print(
    f"FINAL TEST TOP-5 : "
    f"{fused_metrics['top5']:.6f}% "
    f"({fused_metrics['correct_top5']}/211)"
)


# ==================================================================================================
# 30. CONFUSION MATRIX
# ==================================================================================================

confusion = np.zeros(
    (
        NUM_CLASSES,
        NUM_CLASSES
    ),
    dtype=np.int64
)


for target, pred in zip(
    test_targets,
    fused_metrics[
        "pred1"
    ]
):

    confusion[
        int(
            target
        ),
        int(
            pred
        )
    ] += 1


CONFUSION_PATH = (
    REPORT_DIR /
    "l16_confusion_matrix.csv"
)


pd.DataFrame(
    confusion,
    index=[
        id_to_gloss[i]
        for i in range(
            NUM_CLASSES
        )
    ],
    columns=[
        id_to_gloss[i]
        for i in range(
            NUM_CLASSES
        )
    ]
).to_csv(
    CONFUSION_PATH
)


# ==================================================================================================
# 31. PER-SAMPLE REPORT
# ==================================================================================================

rows = []


for i, uid in enumerate(
    test_uids
):

    target = int(
        test_targets[
            i
        ]
    )


    fused_pred = int(
        fused_metrics[
            "pred1"
        ][
            i
        ]
    )


    fused_top5 = (
        fused_metrics[
            "predk"
        ][
            i
        ]
    )


    l12_pred = int(
        l12_metrics[
            "pred1"
        ][
            i
        ]
    )

    l13_pred = int(
        l13_metrics[
            "pred1"
        ][
            i
        ]
    )

    l14_pred = int(
        l14_metrics[
            "pred1"
        ][
            i
        ]
    )


    row = {

        "uid":
            uid,

        "true_class_id":
            target,

        "true_gloss":
            id_to_gloss[
                target
            ],

        "l12b_predicted_class_id":
            l12_pred,

        "l12b_predicted_gloss":
            id_to_gloss[
                l12_pred
            ],

        "l13b_predicted_class_id":
            l13_pred,

        "l13b_predicted_gloss":
            id_to_gloss[
                l13_pred
            ],

        "l14a_predicted_class_id":
            l14_pred,

        "l14a_predicted_gloss":
            id_to_gloss[
                l14_pred
            ],

        "l16_predicted_class_id":
            fused_pred,

        "l16_predicted_gloss":
            id_to_gloss[
                fused_pred
            ],

        "top1_correct":
            bool(
                fused_pred
                ==
                target
            ),

        "top5_correct":
            bool(
                target
                in
                fused_top5
            ),

    }


    for rank in range(
        5
    ):

        cid = int(
            fused_top5[
                rank
            ]
        )

        row[
            f"top{rank+1}_class_id"
        ] = cid

        row[
            f"top{rank+1}_gloss"
        ] = id_to_gloss[
            cid
        ]

        row[
            f"top{rank+1}_fusion_score"
        ] = float(
            FUSED[
                i,
                cid
            ]
        )


    rows.append(
        row
    )


prediction_df = pd.DataFrame(
    rows
)


PREDICTION_PATH = (
    REPORT_DIR /
    "l16_final_test_predictions.csv"
)


prediction_df.to_csv(
    PREDICTION_PATH,
    index=False
)


# ==================================================================================================
# 32. SAVE BRANCH + FUSION SCORES
# ==================================================================================================

SCORE_PATH = (
    MODEL_DIR /
    "l16_final_test_scores.npz"
)


np.savez_compressed(

    SCORE_PATH,

    uid=np.asarray(
        test_uids
    ),

    targets=
        test_targets,

    l12b_probabilities=
        l12_probs,

    l13b_probabilities=
        l13_probs,

    l14a_probabilities=
        l14_probs,

    l12b_sparse_top5=
        S12,

    l13b_sparse_top5=
        S13,

    l14a_sparse_top5=
        S14,

    fused_scores=
        FUSED,

    fused_predictions=
        fused_metrics[
            "pred1"
        ],

    fused_top5=
        fused_metrics[
            "predk"
        ],
)


# ==================================================================================================
# 33. BRANCH COMPARISON
# ==================================================================================================

comparison_df = pd.DataFrame(
    [

        {
            "model":
                "L12B SSL ST-GCN",

            "test_top1":
                l12_metrics[
                    "top1"
                ],

            "test_top5":
                l12_metrics[
                    "top5"
                ],

            "correct_top1":
                l12_metrics[
                    "correct_top1"
                ],

            "correct_top5":
                l12_metrics[
                    "correct_top5"
                ],
        },

        {
            "model":
                "L13B RGB Temporal Transformer",

            "test_top1":
                l13_metrics[
                    "top1"
                ],

            "test_top5":
                l13_metrics[
                    "top5"
                ],

            "correct_top1":
                l13_metrics[
                    "correct_top1"
                ],

            "correct_top5":
                l13_metrics[
                    "correct_top5"
                ],
        },

        {
            "model":
                "L14A VideoMAE",

            "test_top1":
                l14_metrics[
                    "top1"
                ],

            "test_top5":
                l14_metrics[
                    "top5"
                ],

            "correct_top1":
                l14_metrics[
                    "correct_top1"
                ],

            "correct_top5":
                l14_metrics[
                    "correct_top5"
                ],
        },

        {
            "model":
                "L16 Frozen L14C Fusion",

            "test_top1":
                fused_metrics[
                    "top1"
                ],

            "test_top5":
                fused_metrics[
                    "top5"
                ],

            "correct_top1":
                fused_metrics[
                    "correct_top1"
                ],

            "correct_top5":
                fused_metrics[
                    "correct_top5"
                ],
        },

    ]
)


COMPARISON_PATH = (
    REPORT_DIR /
    "l16_test_model_comparison.csv"
)


comparison_df.to_csv(
    COMPARISON_PATH,
    index=False
)


print("\n" + "=" * 140)
print("15. FINAL TEST MODEL COMPARISON")
print("=" * 140)

print(
    comparison_df.to_string(
        index=False
    )
)


# ==================================================================================================
# 34. TIMING REPORT
# ==================================================================================================

timing_report = {

    "L12B": {

        "mean_model_inference_ms":
            float(
                np.mean(
                    l12_times
                )
                *
                1000
            ),

        "median_model_inference_ms":
            float(
                np.median(
                    l12_times
                )
                *
                1000
            ),
    },

    "L13B_temporal_model": {

        "mean_model_inference_ms":
            float(
                np.mean(
                    l13_times
                )
                *
                1000
            ),

        "median_model_inference_ms":
            float(
                np.median(
                    l13_times
                )
                *
                1000
            ),
    },

    "L14A": {

        "mean_model_inference_ms":
            float(
                np.mean(
                    l14_times
                )
                *
                1000
            ),

        "median_model_inference_ms":
            float(
                np.median(
                    l14_times
                )
                *
                1000
            ),
    },
}


TIMING_PATH = (
    REPORT_DIR /
    "l16_timing.json"
)


save_json(
    timing_report,
    TIMING_PATH
)


# ==================================================================================================
# 35. FINAL SUMMARY
# ==================================================================================================

summary = {

    "stage":
        "L16",

    "status":
        "FINAL_TEST_COMPLETE",

    "evaluation_type":
        (
            "One-time final 211-class "
            "test evaluation"
        ),

    "selected_system":
        "L14C",

    "fusion_method":
        (
            "Frozen sparse Top-5 "
            "raw-probability late fusion"
        ),

    "weights": {

        "L12B":
            0.40,

        "L13B":
            0.30,

        "L14A":
            0.30,
    },

    "num_classes":
        211,

    "test_samples":
        211,

    "l12b": {

        "top1":
            l12_metrics[
                "top1"
            ],

        "top5":
            l12_metrics[
                "top5"
            ],
    },

    "l13b": {

        "top1":
            l13_metrics[
                "top1"
            ],

        "top5":
            l13_metrics[
                "top5"
            ],
    },

    "l14a": {

        "top1":
            l14_metrics[
                "top1"
            ],

        "top5":
            l14_metrics[
                "top5"
            ],
    },

    "final_fusion": {

        "top1":
            fused_metrics[
                "top1"
            ],

        "top5":
            fused_metrics[
                "top5"
            ],

        "correct_top1":
            fused_metrics[
                "correct_top1"
            ],

        "correct_top5":
            fused_metrics[
                "correct_top5"
            ],
    },

    "validation_reference": {

        "top1":
            41.232227,

        "top5":
            51.658768,
    },

    "model_selection_changed_after_test":
        False,

    "weights_changed_after_test":
        False,

    "threshold_tuned_on_test":
        False,

    "test_sample_removed":
        False,

    "timestamp":
        datetime.now().isoformat(),
}


SUMMARY_PATH = (
    REPORT_DIR /
    "l16_final_test_summary.json"
)


save_json(
    summary,
    SUMMARY_PATH
)


# ==================================================================================================
# 36. PLOT
# ==================================================================================================

try:

    import matplotlib.pyplot as plt


    labels = [
        "L12B",
        "L13B",
        "L14A",
        "L16 Fusion"
    ]


    top1_values = [
        l12_metrics[
            "top1"
        ],
        l13_metrics[
            "top1"
        ],
        l14_metrics[
            "top1"
        ],
        fused_metrics[
            "top1"
        ],
    ]


    top5_values = [
        l12_metrics[
            "top5"
        ],
        l13_metrics[
            "top5"
        ],
        l14_metrics[
            "top5"
        ],
        fused_metrics[
            "top5"
        ],
    ]


    x = np.arange(
        len(
            labels
        )
    )

    width = 0.36


    plt.figure(
        figsize=(
            10,
            6
        )
    )


    plt.bar(
        x -
        width / 2,
        top1_values,
        width,
        label="Top-1"
    )


    plt.bar(
        x +
        width / 2,
        top5_values,
        width,
        label="Top-5"
    )


    plt.xticks(
        x,
        labels
    )

    plt.ylabel(
        "Final Test Accuracy (%)"
    )

    plt.title(
        "CISLR — Final 211-Class Test Evaluation"
    )

    plt.legend()

    plt.tight_layout()


    plt.savefig(
        PLOT_DIR /
        "l16_final_test_accuracy.png",
        dpi=180
    )


    plt.close()


except Exception as e:

    print(
        "[WARN] Plot skipped:",
        repr(
            e
        )
    )


# ==================================================================================================
# 37. HASH FINAL RESULTS
# ==================================================================================================

result_hashes = {}


for name, path in {

    "predictions":
        PREDICTION_PATH,

    "scores":
        SCORE_PATH,

    "comparison":
        COMPARISON_PATH,

    "confusion":
        CONFUSION_PATH,

    "timing":
        TIMING_PATH,

    "summary":
        SUMMARY_PATH,

}.items():

    result_hashes[
        name
    ] = {

        "path":
            str(
                path
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


HASH_PATH = (
    REPORT_DIR /
    "l16_result_hashes.json"
)


save_json(
    result_hashes,
    HASH_PATH
)


# ==================================================================================================
# 38. FINAL INTEGRITY CHECK
# ==================================================================================================

print("\n" + "=" * 140)
print("16. FINAL L16 INTEGRITY CHECK")
print("=" * 140)

assert l12_probs.shape == (
    211,
    211
)

assert l13_probs.shape == (
    211,
    211
)

assert l14_probs.shape == (
    211,
    211
)

assert FUSED.shape == (
    211,
    211
)

assert np.isfinite(
    l12_probs
).all()

assert np.isfinite(
    l13_probs
).all()

assert np.isfinite(
    l14_probs
).all()

assert np.isfinite(
    FUSED
).all()

assert len(
    prediction_df
) == 211


print(
    "[PASS] 211 test samples evaluated."
)

print(
    "[PASS] L12B probability matrix valid."
)

print(
    "[PASS] L13B probability matrix valid."
)

print(
    "[PASS] L14A probability matrix valid."
)

print(
    "[PASS] Frozen fusion matrix valid."
)

print(
    "[PASS] No training performed."
)

print(
    "[PASS] No fusion-weight search performed."
)

print(
    "[PASS] No threshold tuning performed."
)

print(
    "[PASS] No test samples removed."
)


# ==================================================================================================
# 39. FINAL FREEZE
# ==================================================================================================

print("\n")
print("=" * 140)
print("L16 — FINAL 211-CLASS TEST RESULT")
print("=" * 140)

print()

print(
    f"L12B TEST TOP-1       : "
    f"{l12_metrics['top1']:.6f}% "
    f"({l12_metrics['correct_top1']}/211)"
)

print(
    f"L12B TEST TOP-5       : "
    f"{l12_metrics['top5']:.6f}% "
    f"({l12_metrics['correct_top5']}/211)"
)

print()

print(
    f"L13B TEST TOP-1       : "
    f"{l13_metrics['top1']:.6f}% "
    f"({l13_metrics['correct_top1']}/211)"
)

print(
    f"L13B TEST TOP-5       : "
    f"{l13_metrics['top5']:.6f}% "
    f"({l13_metrics['correct_top5']}/211)"
)

print()

print(
    f"L14A TEST TOP-1       : "
    f"{l14_metrics['top1']:.6f}% "
    f"({l14_metrics['correct_top1']}/211)"
)

print(
    f"L14A TEST TOP-5       : "
    f"{l14_metrics['top5']:.6f}% "
    f"({l14_metrics['correct_top5']}/211)"
)

print()

print("-" * 140)

print(
    f"FINAL FUSION TOP-1    : "
    f"{fused_metrics['top1']:.6f}% "
    f"({fused_metrics['correct_top1']}/211)"
)

print(
    f"FINAL FUSION TOP-5    : "
    f"{fused_metrics['top5']:.6f}% "
    f"({fused_metrics['correct_top5']}/211)"
)

print("-" * 140)

print()

print(
    "Frozen weights        : "
    "L12B=0.40, L13B=0.30, L14A=0.30"
)

print(
    "Validation reference  : "
    "Top-1=41.232227%, Top-5=51.658768%"
)

print()

print(
    "Predictions           :",
    PREDICTION_PATH
)

print(
    "Scores                :",
    SCORE_PATH
)

print(
    "Confusion matrix      :",
    CONFUSION_PATH
)

print(
    "Comparison            :",
    COMPARISON_PATH
)

print(
    "Timing                :",
    TIMING_PATH
)

print(
    "Summary               :",
    SUMMARY_PATH
)

print(
    "Result hashes         :",
    HASH_PATH
)

print()

print(
    "[PASS] L16 ONE-TIME FINAL TEST EVALUATION COMPLETE"
)

print(
    "[LOCK] 211-CLASS TEST RESULT IS NOW FINAL"
)

print(
    "[RULE] DO NOT RETUNE L12B/L13B/L14A/L14C USING THIS TEST RESULT"
)

print(
    "[NEXT] L17 = RESTRICTED-VOCABULARY DEPLOYMENT EXPERIMENT"
)

print("=" * 140)

CISLR — L16
ONE-TIME FINAL 211-CLASS TEST EVALUATION

Python      : 3.11.15
PyTorch     : 2.14.0+cu130
Torchvision : 0.29.0+cu130
Device      : cuda
GPU         : NVIDIA GeForce RTX 4050 Laptop GPU

FROZEN FINAL SYSTEM
--------------------------------------------------------------------------------------------------------------------------------------------
L12B SSL ST-GCN               : 0.40
L13B RGB Temporal Transformer : 0.30
L14A VideoMAE                  : 0.30
Fusion                         : sparse Top-5 raw probability

1. VERIFYING L15 FREEZE
L15 master freeze           : PASS
L15 selection               : PASS
L15 hash manifest           : PASS
L15 test protocol           : PASS
L12B checkpoint             : PASS
L13A checkpoint             : PASS
L13B checkpoint             : PASS
L14A checkpoint             : PASS
L9 test manifest            : PASS
L9 mapping                  : PASS
L6 raw directory            : PASS
RGB video directory         : PASS

[PASS] L15 model sel

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base-finetuned-kinetics and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([400]) in the checkpoint and torch.Size([211]) in the model instantiated
- classifier.weight: found shape torch.Size([400, 768]) in the checkpoint and torch.Size([211, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Missing keys    : []
Unexpected keys : []
[PASS] L14A frozen VideoMAE loaded exactly.

12. L14A FINAL TEST INFERENCE
   10/211
   20/211
   30/211
   40/211
   50/211
   60/211
   70/211
   80/211
   90/211
  100/211
  110/211
  120/211
  130/211
  140/211
  150/211
  160/211
  170/211
  180/211
  190/211
  200/211
  210/211
  211/211

L14A TEST Top-1 : 38.388626% (81/211)
L14A TEST Top-5 : 52.606635% (111/211)

13. APPLYING FROZEN L14C REPRESENTATION
[PASS] All three branches converted to sparse Top-5 raw-probability representation.

14. FROZEN L14C TEST FUSION

Weights:
  L12B = 0.4
  L13B = 0.3
  L14A = 0.3

FINAL TEST TOP-1 : 41.232227% (87/211)
FINAL TEST TOP-5 : 52.132701% (110/211)

15. FINAL TEST MODEL COMPARISON
                        model  test_top1  test_top5  correct_top1  correct_top5
              L12B SSL ST-GCN  26.540284  36.966825            56            78
L13B RGB Temporal Transformer  37.440758  46.919431            79            99
                L14A VideoMAE

In [3]:
# ==================================================================================================
# CISLR — L17
# RESTRICTED-VOCABULARY DEPLOYMENT EXPERIMENT
# 10-SIGN VOCABULARY DISCOVERY + FREEZE + L18 DATA PLAN
# ==================================================================================================
#
# PURPOSE
# -------
# L16 completed the FINAL 211-class research benchmark.
#
# L17 starts a NEW and CLEARLY SEPARATED deployment experiment.
#
# L17 DOES NOT:
#   - retrain the 211-class model
#   - alter L16
#   - select signs using L16 test correctness
#   - use L16 confusion matrix
#   - use L16 test probabilities
#   - claim >90% accuracy
#
# L17 DOES:
#   1. Verify L16 is complete.
#   2. Load CISLR metadata.
#   3. Load L9 TRAIN manifest only for candidate quality/support analysis.
#   4. Discover signs with the strongest available TRAIN-SIDE evidence.
#   5. Rank candidates using NON-TEST criteria:
#        - total dataset support
#        - L9 training support
#        - landmark detection quality
#        - both-hand visibility
#        - low missing-hand rate
#        - low scale-repair burden
#   6. Freeze a 10-sign deployment vocabulary.
#   7. Build deployment class IDs 0..9.
#   8. Create an L18 recording/data-expansion plan.
#   9. Save hashes + master freeze.
#
# IMPORTANT SCIENTIFIC SEPARATION
# -------------------------------
#
# RESEARCH RESULT:
#   211-class CISLR
#   Final L16 test Top-1 = 41.232227%
#
# DEPLOYMENT EXPERIMENT:
#   10 signs
#   New training / validation / deployment-test data will be created later.
#
# The >90% target belongs ONLY to the deployment experiment and must be measured.
#
# ==================================================================================================


# ==================================================================================================
# 0. IMPORTS
# ==================================================================================================

import os
import json
import math
import random
import hashlib
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


# ==================================================================================================
# 1. REPRODUCIBILITY
# ==================================================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# ==================================================================================================
# 2. PROJECT PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)

DATASET_CSV = Path(
    "/home/dipshikha/Music/cao/dataset.csv"
)

VIDEO_DIR = Path(
    "/home/dipshikha/Music/cao/CISLR_v1.5-a_videos"
)


# --------------------------------------------------------------------------------------------------
# L9
# --------------------------------------------------------------------------------------------------

L9_ROOT = (
    ROOT /
    "audit/l9_split_freeze"
)

L9_TRAIN = (
    L9_ROOT /
    "splits/l9_train_manifest.csv"
)

L9_VAL = (
    L9_ROOT /
    "splits/l9_validation_manifest.csv"
)

L9_TEST = (
    L9_ROOT /
    "splits/l9_test_manifest.csv"
)

L9_MAPPING = (
    L9_ROOT /
    "class_mapping/l9_controlled_class_mapping.json"
)


# --------------------------------------------------------------------------------------------------
# L16
# --------------------------------------------------------------------------------------------------

L16_ROOT = (
    ROOT /
    "audit/l16_final_test"
)

L16_SUMMARY = (
    L16_ROOT /
    "reports/l16_final_test_summary.json"
)


# --------------------------------------------------------------------------------------------------
# L17 OUTPUT
# --------------------------------------------------------------------------------------------------

L17_ROOT = (
    ROOT /
    "audit/l17_deployment_vocabulary"
)

REPORT_DIR = (
    L17_ROOT /
    "reports"
)

VOCAB_DIR = (
    L17_ROOT /
    "vocabulary"
)

PLAN_DIR = (
    L17_ROOT /
    "l18_data_plan"
)

HASH_DIR = (
    L17_ROOT /
    "hashes"
)

for path in [

    L17_ROOT,
    REPORT_DIR,
    VOCAB_DIR,
    PLAN_DIR,
    HASH_DIR,

]:

    path.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 3. CONFIGURATION
# ==================================================================================================

NUM_DEPLOYMENT_SIGNS = 10


# Desired NEW recording target for L18.
#
# This is intentionally much larger than the original CISLR support.
# We want a deployment model with meaningful signer/background variation.

TARGET_NEW_CLIPS_PER_SIGN = 60


# Recommended L18 split of NEW recordings:
#
# 40 train
# 10 validation
# 10 deployment test
#
# per sign.
#
# Total:
#   400 train
#   100 validation
#   100 deployment test
#
# Original CISLR examples may later be added to TRAIN only
# after L18 protocol is frozen.

NEW_TRAIN_PER_SIGN = 40
NEW_VAL_PER_SIGN = 10
NEW_TEST_PER_SIGN = 10


assert (
    NEW_TRAIN_PER_SIGN
    +
    NEW_VAL_PER_SIGN
    +
    NEW_TEST_PER_SIGN
    ==
    TARGET_NEW_CLIPS_PER_SIGN
)


# ==================================================================================================
# 4. HELPERS
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def save_json(
    obj,
    path
):

    with open(
        path,
        "w"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False
        )


def safe_numeric(
    series
):

    return pd.to_numeric(
        series,
        errors="coerce"
    )


def minmax(
    series
):

    x = safe_numeric(
        series
    ).astype(
        float
    )

    finite = np.isfinite(
        x
    )

    result = pd.Series(
        np.zeros(
            len(x),
            dtype=np.float64
        ),
        index=x.index
    )

    if not finite.any():

        return result


    lo = float(
        x[
            finite
        ].min()
    )

    hi = float(
        x[
            finite
        ].max()
    )


    if abs(
        hi -
        lo
    ) < 1e-12:

        result.loc[
            finite
        ] = 1.0

        return result


    result.loc[
        finite
    ] = (

        x[
            finite
        ]
        -
        lo

    ) / (
        hi -
        lo
    )


    return result


def first_existing_column(
    df,
    candidates
):

    for col in candidates:

        if col in df.columns:

            return col

    return None


# ==================================================================================================
# 5. HEADER
# ==================================================================================================

print("=" * 140)
print("CISLR — L17")
print("10-SIGN RESTRICTED-VOCABULARY DEPLOYMENT EXPERIMENT")
print("=" * 140)

print()
print(
    "Python       :",
    platform.python_version()
)

print(
    "Random seed  :",
    SEED
)

print(
    "Vocabulary   :",
    NUM_DEPLOYMENT_SIGNS,
    "signs"
)

print(
    "L18 target   :",
    TARGET_NEW_CLIPS_PER_SIGN,
    "NEW clips/sign"
)

print()
print(
    "IMPORTANT:"
)

print(
    "L17 will NOT use L16 test predictions, "
    "confusion matrix, or per-class test correctness."
)


# ==================================================================================================
# 6. REQUIRED FILE CHECK
# ==================================================================================================

print("\n" + "=" * 140)
print("1. REQUIRED ARTIFACT CHECK")
print("=" * 140)

required = {

    "dataset.csv":
        DATASET_CSV,

    "L9 train manifest":
        L9_TRAIN,

    "L9 validation manifest":
        L9_VAL,

    "L9 test manifest":
        L9_TEST,

    "L9 mapping":
        L9_MAPPING,

    "L16 final summary":
        L16_SUMMARY,

    "CISLR video directory":
        VIDEO_DIR,
}


for name, path in required.items():

    exists = path.exists()

    print(
        f"{name:28s}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    if not exists:

        raise FileNotFoundError(
            path
        )


print()
print(
    "[PASS] Required frozen artifacts exist."
)


# ==================================================================================================
# 7. VERIFY L16 IS FINAL
#
# We read ONLY the summary to verify that the research benchmark is complete.
#
# We DO NOT load:
#   l16_final_test_predictions.csv
#   l16_confusion_matrix.csv
#   l16_final_test_scores.npz
#
# ==================================================================================================

print("\n" + "=" * 140)
print("2. VERIFYING L16 RESEARCH FREEZE")
print("=" * 140)

with open(
    L16_SUMMARY,
    "r"
) as f:

    l16_summary = json.load(
        f
    )


if (
    l16_summary.get(
        "status"
    )
    !=
    "FINAL_TEST_COMPLETE"
):

    raise RuntimeError(
        "L16 final test is not marked complete."
    )


research_top1 = float(
    l16_summary[
        "final_fusion"
    ][
        "top1"
    ]
)

research_top5 = float(
    l16_summary[
        "final_fusion"
    ][
        "top5"
    ]
)


print(
    "Research benchmark classes : 211"
)

print(
    f"Frozen L16 Test Top-1      : "
    f"{research_top1:.6f}%"
)

print(
    f"Frozen L16 Test Top-5      : "
    f"{research_top5:.6f}%"
)

print()
print(
    "[PASS] L16 remains historical/frozen."
)

print(
    "[PASS] No L16 per-sample predictions loaded."
)

print(
    "[PASS] No L16 confusion matrix loaded."
)

print(
    "[PASS] No L16 score matrix loaded."
)


# ==================================================================================================
# 8. LOAD ORIGINAL CISLR METADATA
# ==================================================================================================

print("\n" + "=" * 140)
print("3. ORIGINAL CISLR METADATA")
print("=" * 140)

dataset_df = pd.read_csv(
    DATASET_CSV
)


print(
    "Rows:",
    len(
        dataset_df
    )
)

print(
    "Columns:",
    list(
        dataset_df.columns
    )
)


dataset_gloss_col = (
    first_existing_column(
        dataset_df,
        [
            "gloss",
            "authoritative_gloss",
            "feature_gloss"
        ]
    )
)


dataset_uid_col = (
    first_existing_column(
        dataset_df,
        [
            "uid",
            "uid_clean"
        ]
    )
)


if dataset_gloss_col is None:

    raise RuntimeError(
        "Could not identify gloss column in dataset.csv."
    )


if dataset_uid_col is None:

    raise RuntimeError(
        "Could not identify UID column in dataset.csv."
    )


dataset_df[
    dataset_gloss_col
] = (
    dataset_df[
        dataset_gloss_col
    ]
    .astype(str)
    .str.strip()
)


dataset_df[
    dataset_uid_col
] = (
    dataset_df[
        dataset_uid_col
    ]
    .astype(str)
    .str.strip()
)


# Remove obviously invalid stringified missing labels.

invalid_gloss_strings = {

    "",
    "nan",
    "none",
    "null",
    "None",
    "NaN",
}


dataset_valid = (
    dataset_df[
        ~dataset_df[
            dataset_gloss_col
        ].isin(
            invalid_gloss_strings
        )
    ]
    .copy()
)


dataset_support = (

    dataset_valid
    .groupby(
        dataset_gloss_col
    )
    .size()
    .rename(
        "dataset_total_support"
    )
    .reset_index()
    .rename(
        columns={
            dataset_gloss_col:
                "gloss"
        }
    )
)


print(
    "Valid metadata rows:",
    len(
        dataset_valid
    )
)

print(
    "Unique valid glosses:",
    dataset_support[
        "gloss"
    ].nunique()
)


# ==================================================================================================
# 9. LOAD L9 TRAIN ONLY FOR VOCABULARY SELECTION
# ==================================================================================================

print("\n" + "=" * 140)
print("4. L9 TRAIN-SIDE CANDIDATE DATA")
print("=" * 140)

train_df = pd.read_csv(
    L9_TRAIN
)


print(
    "Train rows:",
    len(
        train_df
    )
)

print(
    "Train classes:",
    train_df[
        "gloss"
    ].nunique()
)


if len(
    train_df
) != 585:

    raise RuntimeError(
        f"Expected 585 L9 training rows, got {len(train_df)}."
    )


if train_df[
    "gloss"
].nunique() != 211:

    raise RuntimeError(
        "Expected 211 classes in L9 training manifest."
    )


if (
    "split"
    in train_df.columns
):

    split_values = set(

        train_df[
            "split"
        ]
        .astype(str)
        .str.lower()
        .unique()
    )


    if split_values != {
        "train"
    }:

        raise RuntimeError(
            f"Unexpected L9 train split values: {split_values}"
        )


print()
print(
    "[PASS] Candidate analysis uses L9 TRAIN rows."
)


# ==================================================================================================
# 10. DETERMINE AVAILABLE QUALITY COLUMNS
# ==================================================================================================

quality_columns = {

    "original_detection_rate":
        first_existing_column(
            train_df,
            [
                "original_detection_rate"
            ]
        ),

    "repaired_detection_rate":
        first_existing_column(
            train_df,
            [
                "repaired_detection_rate"
            ]
        ),

    "sampled_detection_rate":
        first_existing_column(
            train_df,
            [
                "sampled_detection_rate"
            ]
        ),

    "both_hand_rate":
        first_existing_column(
            train_df,
            [
                "both_hand_rate"
            ]
        ),

    "zero_hand_rate":
        first_existing_column(
            train_df,
            [
                "zero_hand_rate"
            ]
        ),

    "scale_repairs":
        first_existing_column(
            train_df,
            [
                "scale_repairs"
            ]
        ),

    "max_abs_feature":
        first_existing_column(
            train_df,
            [
                "max_abs_feature"
            ]
        ),
}


print()
print(
    "Available quality signals:"
)

for key, value in (
    quality_columns.items()
):

    print(
        f"  {key:26s}: "
        f"{'YES' if value else 'NO'}"
    )


# ==================================================================================================
# 11. AGGREGATE TRAIN-SIDE CLASS STATISTICS
# ==================================================================================================

print("\n" + "=" * 140)
print("5. BUILDING NON-TEST CLASS STATISTICS")
print("=" * 140)


agg_spec = {

    "train_support":
        (
            "uid",
            "count"
        )
}


if (
    quality_columns[
        "original_detection_rate"
    ]
):

    agg_spec[
        "mean_original_detection"
    ] = (
        quality_columns[
            "original_detection_rate"
        ],
        "mean"
    )


if (
    quality_columns[
        "repaired_detection_rate"
    ]
):

    agg_spec[
        "mean_repaired_detection"
    ] = (
        quality_columns[
            "repaired_detection_rate"
        ],
        "mean"
    )


if (
    quality_columns[
        "sampled_detection_rate"
    ]
):

    agg_spec[
        "mean_sampled_detection"
    ] = (
        quality_columns[
            "sampled_detection_rate"
        ],
        "mean"
    )


if (
    quality_columns[
        "both_hand_rate"
    ]
):

    agg_spec[
        "mean_both_hand_rate"
    ] = (
        quality_columns[
            "both_hand_rate"
        ],
        "mean"
    )


if (
    quality_columns[
        "zero_hand_rate"
    ]
):

    agg_spec[
        "mean_zero_hand_rate"
    ] = (
        quality_columns[
            "zero_hand_rate"
        ],
        "mean"
    )


if (
    quality_columns[
        "scale_repairs"
    ]
):

    agg_spec[
        "mean_scale_repairs"
    ] = (
        quality_columns[
            "scale_repairs"
        ],
        "mean"
    )


if (
    quality_columns[
        "max_abs_feature"
    ]
):

    agg_spec[
        "mean_max_abs_feature"
    ] = (
        quality_columns[
            "max_abs_feature"
        ],
        "mean"
    )


class_stats = (

    train_df
    .groupby(
        "gloss"
    )
    .agg(
        **agg_spec
    )
    .reset_index()
)


# Add original total support.

class_stats = (
    class_stats
    .merge(
        dataset_support,
        on="gloss",
        how="left"
    )
)


class_stats[
    "dataset_total_support"
] = (

    class_stats[
        "dataset_total_support"
    ]
    .fillna(
        class_stats[
            "train_support"
        ]
    )
    .astype(int)
)


# ==================================================================================================
# 12. BUILD SELECTION SCORE
#
# This score is deliberately based on NON-TEST information.
#
# Primary signal:
#   dataset support
#   train support
#
# Secondary signal:
#   hand detection / visibility quality
#
# It is NOT a prediction-performance ranking.
# ==================================================================================================

class_stats[
    "support_score"
] = (
    minmax(
        class_stats[
            "dataset_total_support"
        ]
    )
)


class_stats[
    "train_support_score"
] = (
    minmax(
        class_stats[
            "train_support"
        ]
    )
)


# Default neutral values.

class_stats[
    "detection_score"
] = 0.5

class_stats[
    "both_hand_score"
] = 0.5

class_stats[
    "missing_penalty_score"
] = 0.5

class_stats[
    "repair_penalty_score"
] = 0.5


if (
    "mean_original_detection"
    in class_stats.columns
):

    class_stats[
        "detection_score"
    ] = minmax(
        class_stats[
            "mean_original_detection"
        ]
    )


elif (
    "mean_repaired_detection"
    in class_stats.columns
):

    class_stats[
        "detection_score"
    ] = minmax(
        class_stats[
            "mean_repaired_detection"
        ]
    )


if (
    "mean_both_hand_rate"
    in class_stats.columns
):

    class_stats[
        "both_hand_score"
    ] = minmax(
        class_stats[
            "mean_both_hand_rate"
        ]
    )


if (
    "mean_zero_hand_rate"
    in class_stats.columns
):

    class_stats[
        "missing_penalty_score"
    ] = (

        1.0
        -
        minmax(
            class_stats[
                "mean_zero_hand_rate"
            ]
        )
    )


if (
    "mean_scale_repairs"
    in class_stats.columns
):

    class_stats[
        "repair_penalty_score"
    ] = (

        1.0
        -
        minmax(
            class_stats[
                "mean_scale_repairs"
            ]
        )
    )


# --------------------------------------------------------------------------------------------------
# Fixed L17 ranking weights.
#
# Support is emphasized because the existing CISLR dataset is extremely sparse.
#
# --------------------------------------------------------------------------------------------------

class_stats[
    "l17_selection_score"
] = (

    0.35
    *
    class_stats[
        "support_score"
    ]

    +

    0.25
    *
    class_stats[
        "train_support_score"
    ]

    +

    0.20
    *
    class_stats[
        "detection_score"
    ]

    +

    0.10
    *
    class_stats[
        "both_hand_score"
    ]

    +

    0.05
    *
    class_stats[
        "missing_penalty_score"
    ]

    +

    0.05
    *
    class_stats[
        "repair_penalty_score"
    ]
)


# Stable deterministic ordering.

class_stats = (
    class_stats
    .sort_values(
        [
            "l17_selection_score",
            "dataset_total_support",
            "train_support",
            "gloss"
        ],
        ascending=[
            False,
            False,
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


class_stats[
    "l17_rank"
] = (
    np.arange(
        1,
        len(
            class_stats
        )
        +
        1
    )
)


# ==================================================================================================
# 13. SAVE FULL NON-TEST CANDIDATE RANKING
# ==================================================================================================

CANDIDATE_PATH = (
    REPORT_DIR /
    "l17_non_test_candidate_ranking.csv"
)


class_stats.to_csv(
    CANDIDATE_PATH,
    index=False
)


print(
    "Candidate classes:",
    len(
        class_stats
    )
)

print()
print(
    "Top 25 candidates:"
)


display_cols = [

    "l17_rank",
    "gloss",
    "dataset_total_support",
    "train_support",
]


for col in [

    "mean_original_detection",
    "mean_both_hand_rate",
    "mean_zero_hand_rate",
    "l17_selection_score",

]:

    if col in class_stats.columns:

        display_cols.append(
            col
        )


print(
    class_stats[
        display_cols
    ]
    .head(
        25
    )
    .to_string(
        index=False
    )
)


# ==================================================================================================
# 14. FREEZE 10-SIGN VOCABULARY
#
# IMPORTANT:
#
# This is an AUTOMATIC DATA-DRIVEN vocabulary based only on:
#   - original dataset support
#   - L9 TRAIN support
#   - L9 TRAIN landmark quality
#
# NOT on L16 test correctness.
#
# ==================================================================================================

selected = (
    class_stats
    .head(
        NUM_DEPLOYMENT_SIGNS
    )
    .copy()
)


selected[
    "deployment_class_id"
] = np.arange(
    NUM_DEPLOYMENT_SIGNS,
    dtype=np.int64
)


selected[
    "selection_basis"
] = (
    "non_test_support_and_landmark_quality"
)


selected[
    "selected_in_l17"
] = True


# Put deployment ID first.

front_cols = [

    "deployment_class_id",
    "gloss",
    "dataset_total_support",
    "train_support",
    "l17_rank",
    "l17_selection_score",
    "selection_basis",
]


remaining_cols = [

    c
    for c in selected.columns
    if c not in front_cols
]


selected = selected[
    front_cols
    +
    remaining_cols
]


# ==================================================================================================
# 15. PRINT FROZEN VOCABULARY
# ==================================================================================================

print("\n" + "=" * 140)
print("6. SELECTED 10-SIGN DEPLOYMENT VOCABULARY")
print("=" * 140)

print()

print(
    selected[
        [
            "deployment_class_id",
            "gloss",
            "dataset_total_support",
            "train_support",
            "l17_selection_score"
        ]
    ]
    .to_string(
        index=False
    )
)


# ==================================================================================================
# 16. VOCABULARY INTEGRITY
# ==================================================================================================

assert len(
    selected
) == 10

assert selected[
    "gloss"
].nunique() == 10

assert set(
    selected[
        "deployment_class_id"
    ].tolist()
) == set(
    range(
        10
    )
)


selected_glosses = (
    selected[
        "gloss"
    ]
    .astype(str)
    .tolist()
)


print()
print(
    "[PASS] Exactly 10 deployment signs."
)

print(
    "[PASS] Deployment IDs are 0..9."
)

print(
    "[PASS] No duplicate glosses."
)


# ==================================================================================================
# 17. SAVE VOCABULARY
# ==================================================================================================

VOCAB_CSV = (
    VOCAB_DIR /
    "l17_10_sign_vocabulary.csv"
)


selected.to_csv(
    VOCAB_CSV,
    index=False
)


VOCAB_JSON = (
    VOCAB_DIR /
    "l17_10_sign_vocabulary.json"
)


vocab_json = {

    "stage":
        "L17",

    "experiment":
        (
            "restricted_vocabulary_"
            "deployment"
        ),

    "num_classes":
        10,

    "selection_seed":
        SEED,

    "selection_basis":
        (
            "Original CISLR support + "
            "L9 TRAIN support + "
            "L9 TRAIN landmark quality. "
            "No L16 per-sample test performance."
        ),

    "deployment_class_to_gloss": {

        str(
            int(
                row[
                    "deployment_class_id"
                ]
            )
        ):
            str(
                row[
                    "gloss"
                ]
            )

        for _, row
        in selected.iterrows()
    },

    "gloss_to_deployment_class": {

        str(
            row[
                "gloss"
            ]
        ):
            int(
                row[
                    "deployment_class_id"
                ]
            )

        for _, row
        in selected.iterrows()
    },

    "signs": [

        {

            "deployment_class_id":
                int(
                    row[
                        "deployment_class_id"
                    ]
                ),

            "gloss":
                str(
                    row[
                        "gloss"
                    ]
                ),

            "dataset_total_support":
                int(
                    row[
                        "dataset_total_support"
                    ]
                ),

            "l9_train_support":
                int(
                    row[
                        "train_support"
                    ]
                ),

            "l17_rank":
                int(
                    row[
                        "l17_rank"
                    ]
                ),

            "selection_score":
                float(
                    row[
                        "l17_selection_score"
                    ]
                ),
        }

        for _, row
        in selected.iterrows()
    ],
}


save_json(
    vocab_json,
    VOCAB_JSON
)


# ==================================================================================================
# 18. CREATE EXISTING CISLR TRAIN-ONLY SUBSET
#
# IMPORTANT:
#
# This is NOT the final deployment train/validation/test split.
#
# It simply records the existing L9 TRAIN samples belonging to the
# selected 10 signs.
#
# L9 validation/test are deliberately NOT copied into deployment training.
#
# ==================================================================================================

existing_train = (
    train_df[
        train_df[
            "gloss"
        ].isin(
            selected_glosses
        )
    ]
    .copy()
)


gloss_to_deployment_id = {

    str(
        row[
            "gloss"
        ]
    ):
        int(
            row[
                "deployment_class_id"
            ]
        )

    for _, row
    in selected.iterrows()
}


existing_train[
    "deployment_class_id"
] = (
    existing_train[
        "gloss"
    ]
    .map(
        gloss_to_deployment_id
    )
    .astype(int)
)


existing_train[
    "deployment_source"
] = (
    "CISLR_L9_TRAIN_ONLY"
)


existing_train[
    "deployment_usage"
] = (
    "eligible_for_training_only"
)


EXISTING_TRAIN_PATH = (
    PLAN_DIR /
    "l17_existing_cislr_train_only.csv"
)


existing_train.to_csv(
    EXISTING_TRAIN_PATH,
    index=False
)


print("\n" + "=" * 140)
print("7. EXISTING CISLR TRAIN-ONLY MATERIAL")
print("=" * 140)

existing_counts = (

    existing_train
    .groupby(
        [
            "deployment_class_id",
            "gloss"
        ]
    )
    .size()
    .rename(
        "existing_l9_train_clips"
    )
    .reset_index()
)


print(
    existing_counts.to_string(
        index=False
    )
)

print()
print(
    "Existing eligible CISLR training clips:",
    len(
        existing_train
    )
)


# ==================================================================================================
# 19. BUILD L18 NEW RECORDING PLAN
# ==================================================================================================

print("\n" + "=" * 140)
print("8. L18 DATA-EXPANSION PLAN")
print("=" * 140)


plan_rows = []


for _, row in (
    selected.iterrows()
):

    deployment_id = int(
        row[
            "deployment_class_id"
        ]
    )

    gloss = str(
        row[
            "gloss"
        ]
    )


    for clip_index in range(
        TARGET_NEW_CLIPS_PER_SIGN
    ):

        if (
            clip_index
            <
            NEW_TRAIN_PER_SIGN
        ):

            split = (
                "train"
            )

        elif (
            clip_index
            <
            NEW_TRAIN_PER_SIGN
            +
            NEW_VAL_PER_SIGN
        ):

            split = (
                "validation"
            )

        else:

            split = (
                "deployment_test"
            )


        recording_id = (

            f"D{deployment_id:02d}_"
            f"{gloss}_"
            f"{clip_index+1:03d}"
        )


        plan_rows.append(
            {

                "recording_id":
                    recording_id,

                "deployment_class_id":
                    deployment_id,

                "gloss":
                    gloss,

                "planned_split":
                    split,

                "new_recording":
                    True,

                "recorded":
                    False,

                "video_path":
                    "",

                "signer_id":
                    "",

                "session_id":
                    "",

                "background_id":
                    "",

                "lighting_id":
                    "",

                "distance_id":
                    "",

                "quality_checked":
                    False,

                "notes":
                    "",
            }
        )


recording_plan = pd.DataFrame(
    plan_rows
)


PLAN_PATH = (
    PLAN_DIR /
    "l18_new_recording_plan.csv"
)


recording_plan.to_csv(
    PLAN_PATH,
    index=False
)


# ==================================================================================================
# 20. VERIFY L18 SPLIT COUNTS
# ==================================================================================================

split_counts = (

    recording_plan
    .groupby(
        [
            "gloss",
            "planned_split"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


print(
    split_counts.to_string()
)


for gloss in (
    selected_glosses
):

    row = split_counts.loc[
        gloss
    ]


    assert int(
        row.get(
            "train",
            0
        )
    ) == NEW_TRAIN_PER_SIGN


    assert int(
        row.get(
            "validation",
            0
        )
    ) == NEW_VAL_PER_SIGN


    assert int(
        row.get(
            "deployment_test",
            0
        )
    ) == NEW_TEST_PER_SIGN


assert len(
    recording_plan
) == (
    10
    *
    60
)


print()
print(
    "[PASS] Planned NEW recordings:",
    len(
        recording_plan
    )
)

print(
    "[PASS] NEW train clips:",
    int(
        (
            recording_plan[
                "planned_split"
            ]
            ==
            "train"
        ).sum()
    )
)

print(
    "[PASS] NEW validation clips:",
    int(
        (
            recording_plan[
                "planned_split"
            ]
            ==
            "validation"
        ).sum()
    )
)

print(
    "[PASS] NEW deployment-test clips:",
    int(
        (
            recording_plan[
                "planned_split"
            ]
            ==
            "deployment_test"
        ).sum()
    )
)


# ==================================================================================================
# 21. CREATE DIRECTORY TEMPLATE FOR L18
# ==================================================================================================

DATA_COLLECTION_ROOT = (
    PROJECT_ROOT /
    "deployment_10_sign_data"
)


for _, row in (
    selected.iterrows()
):

    deployment_id = int(
        row[
            "deployment_class_id"
        ]
    )

    gloss = str(
        row[
            "gloss"
        ]
    )


    class_folder = (

        DATA_COLLECTION_ROOT /
        f"{deployment_id:02d}_{gloss}"
    )


    for split in [

        "train",
        "validation",
        "deployment_test",

    ]:

        (
            class_folder /
            split
        ).mkdir(
            parents=True,
            exist_ok=True
        )


print()
print(
    "[PASS] L18 directory template created:"
)

print(
    DATA_COLLECTION_ROOT
)


# ==================================================================================================
# 22. CREATE README FOR DATA COLLECTION
# ==================================================================================================

README_PATH = (
    PLAN_DIR /
    "L18_RECORDING_PROTOCOL.txt"
)


readme_text = f"""
CISLR — L18 DEPLOYMENT DATA COLLECTION PROTOCOL
================================================

L17 vocabulary size:
{NUM_DEPLOYMENT_SIGNS}

NEW recordings/sign:
{TARGET_NEW_CLIPS_PER_SIGN}

NEW split/sign:
TRAIN      = {NEW_TRAIN_PER_SIGN}
VALIDATION = {NEW_VAL_PER_SIGN}
TEST       = {NEW_TEST_PER_SIGN}

TOTAL NEW RECORDINGS:
{NUM_DEPLOYMENT_SIGNS * TARGET_NEW_CLIPS_PER_SIGN}

TARGET:
Build a high-reliability restricted-vocabulary sign-language
deployment model.

IMPORTANT:
The >90% accuracy objective is a TARGET, not a guaranteed result.

DATA COLLECTION RULES
---------------------

1. Record the COMPLETE sign gesture.

2. Start with hands in a natural/rest position.

3. Perform the sign.

4. Return toward a neutral/rest position.

5. Keep the signer visible.

6. Do not crop hands out of the image.

7. Record variations in:
   - distance
   - lighting
   - background
   - clothing
   - speed
   - hand position

8. Prefer multiple recording sessions.

9. Prefer multiple signers if available.

10. Validation and deployment-test clips must not be duplicates
    or near-duplicates of training clips.

11. Do not move difficult validation/test clips into training
    after observing model errors.

12. Original L16 test results must NOT be used for tuning this
    deployment experiment.

13. Original CISLR L9 TRAIN examples for the selected signs may
    later be incorporated into deployment TRAINING.

14. Original L9 validation/test samples should NOT be silently
    reused as the new deployment validation/test benchmark.

15. NO_SIGN / background examples are added later in L20.

DEPLOYMENT CLASSES
------------------

"""


for _, row in (
    selected.iterrows()
):

    readme_text += (

        f"{int(row['deployment_class_id']):02d} "
        f"{row['gloss']}\n"
    )


with open(
    README_PATH,
    "w"
) as f:

    f.write(
        readme_text
    )


# ==================================================================================================
# 23. EXPERIMENT SEPARATION RECORD
# ==================================================================================================

separation_record = {

    "research_experiment": {

        "name":
            "211-class CISLR research benchmark",

        "final_stage":
            "L16",

        "num_classes":
            211,

        "final_test_top1":
            research_top1,

        "final_test_top5":
            research_top5,

        "status":
            "FROZEN",

        "may_be_retuned":
            False,
    },


    "deployment_experiment": {

        "name":
            "10-sign restricted-vocabulary deployment",

        "starting_stage":
            "L17",

        "num_classes":
            10,

        "accuracy_target":
            ">=90% validation if empirically achieved",

        "target_is_guarantee":
            False,

        "requires_new_data":
            True,

        "new_recordings_per_sign":
            TARGET_NEW_CLIPS_PER_SIGN,

        "planned_new_train":
            NUM_DEPLOYMENT_SIGNS
            *
            NEW_TRAIN_PER_SIGN,

        "planned_new_validation":
            NUM_DEPLOYMENT_SIGNS
            *
            NEW_VAL_PER_SIGN,

        "planned_new_test":
            NUM_DEPLOYMENT_SIGNS
            *
            NEW_TEST_PER_SIGN,
    },


    "anti_leakage": {

        "l16_predictions_used_for_vocab_selection":
            False,

        "l16_confusion_used_for_vocab_selection":
            False,

        "l16_test_correctness_used_for_vocab_selection":
            False,

        "l16_scores_used_for_vocab_selection":
            False,

        "l9_train_support_used":
            True,

        "l9_train_quality_used":
            True,

        "dataset_metadata_support_used":
            True,
    },
}


SEPARATION_PATH = (
    REPORT_DIR /
    "l17_experiment_separation.json"
)


save_json(
    separation_record,
    SEPARATION_PATH
)


# ==================================================================================================
# 24. SAVE L17 SUMMARY
# ==================================================================================================

summary = {

    "stage":
        "L17",

    "status":
        "COMPLETE_AND_FROZEN",

    "experiment":
        "10-sign restricted-vocabulary deployment",

    "num_deployment_classes":
        NUM_DEPLOYMENT_SIGNS,

    "selection_seed":
        SEED,

    "selection_basis":
        (
            "Dataset support + L9 TRAIN support + "
            "L9 TRAIN landmark quality"
        ),

    "l16_per_sample_test_information_used":
        False,

    "selected_glosses":
        selected_glosses,

    "deployment_mapping": {

        str(
            int(
                row[
                    "deployment_class_id"
                ]
            )
        ):
            str(
                row[
                    "gloss"
                ]
            )

        for _, row
        in selected.iterrows()
    },

    "existing_l9_train_clips":
        int(
            len(
                existing_train
            )
        ),

    "l18_new_recordings_per_sign":
        TARGET_NEW_CLIPS_PER_SIGN,

    "l18_total_new_recordings":
        int(
            len(
                recording_plan
            )
        ),

    "l18_new_train_clips":
        int(
            (
                recording_plan[
                    "planned_split"
                ]
                ==
                "train"
            ).sum()
        ),

    "l18_new_validation_clips":
        int(
            (
                recording_plan[
                    "planned_split"
                ]
                ==
                "validation"
            ).sum()
        ),

    "l18_new_test_clips":
        int(
            (
                recording_plan[
                    "planned_split"
                ]
                ==
                "deployment_test"
            ).sum()
        ),

    "research_reference": {

        "l16_num_classes":
            211,

        "l16_test_top1":
            research_top1,

        "l16_test_top5":
            research_top5,

        "status":
            "frozen_do_not_retune",
    },

    "accuracy_target": {

        "deployment_validation_target":
            ">=90%",

        "guaranteed":
            False,

        "must_be_measured":
            True,
    },

    "next_stage":
        (
            "L18 — deployment data expansion "
            "and recording"
        ),

    "timestamp":
        datetime.now().isoformat(),
}


SUMMARY_PATH = (
    REPORT_DIR /
    "l17_summary.json"
)


save_json(
    summary,
    SUMMARY_PATH
)


# ==================================================================================================
# 25. HASH IMPORTANT L17 ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 140)
print("9. HASHING L17 FREEZE ARTIFACTS")
print("=" * 140)


artifact_paths = {

    "vocabulary_csv":
        VOCAB_CSV,

    "vocabulary_json":
        VOCAB_JSON,

    "candidate_ranking":
        CANDIDATE_PATH,

    "existing_train_subset":
        EXISTING_TRAIN_PATH,

    "recording_plan":
        PLAN_PATH,

    "recording_protocol":
        README_PATH,

    "experiment_separation":
        SEPARATION_PATH,

    "summary":
        SUMMARY_PATH,
}


hash_manifest = {}


for name, path in (
    artifact_paths.items()
):

    digest = sha256_file(
        path
    )


    hash_manifest[
        name
    ] = {

        "path":
            str(
                path
            ),

        "sha256":
            digest,
    }


    print(
        f"{name:25s}: "
        f"{digest}"
    )


HASH_MANIFEST_PATH = (
    HASH_DIR /
    "l17_sha256_manifest.json"
)


save_json(
    hash_manifest,
    HASH_MANIFEST_PATH
)


# ==================================================================================================
# 26. MASTER FREEZE
# ==================================================================================================

master_freeze = {

    "stage":
        "L17",

    "status":
        "COMPLETE_AND_FROZEN",

    "experiment":
        (
            "restricted_vocabulary_"
            "deployment"
        ),

    "num_classes":
        10,

    "vocabulary":
        [

            {

                "deployment_class_id":
                    int(
                        row[
                            "deployment_class_id"
                        ]
                    ),

                "gloss":
                    str(
                        row[
                            "gloss"
                        ]
                    ),

            }

            for _, row
            in selected.iterrows()
        ],

    "selection_basis":
        (
            "non-test dataset support "
            "and L9 train quality"
        ),

    "l16_test_predictions_used":
        False,

    "l16_test_confusion_used":
        False,

    "l16_test_scores_used":
        False,

    "target_new_clips_per_sign":
        TARGET_NEW_CLIPS_PER_SIGN,

    "planned_new_split_per_sign": {

        "train":
            NEW_TRAIN_PER_SIGN,

        "validation":
            NEW_VAL_PER_SIGN,

        "deployment_test":
            NEW_TEST_PER_SIGN,
    },

    "next_stage":
        "L18",

    "created_at":
        datetime.now().isoformat(),
}


MASTER_FREEZE_PATH = (
    L17_ROOT /
    "L17_MASTER_FREEZE.json"
)


save_json(
    master_freeze,
    MASTER_FREEZE_PATH
)


MASTER_SHA256 = sha256_file(
    MASTER_FREEZE_PATH
)


# ==================================================================================================
# 27. FINAL ASSERTIONS
# ==================================================================================================

print("\n" + "=" * 140)
print("10. FINAL L17 INTEGRITY CHECK")
print("=" * 140)


checks = {

    "exactly_10_signs":
        len(
            selected
        ) == 10,

    "unique_glosses":
        selected[
            "gloss"
        ].nunique() == 10,

    "deployment_ids_0_to_9":
        set(
            selected[
                "deployment_class_id"
            ]
            .tolist()
        ) == set(
            range(
                10
            )
        ),

    "l16_predictions_not_loaded":
        True,

    "l16_confusion_not_loaded":
        True,

    "l16_scores_not_loaded":
        True,

    "existing_data_train_only":
        (
            existing_train[
                "deployment_usage"
            ]
            ==
            "eligible_for_training_only"
        ).all(),

    "600_new_clips_planned":
        len(
            recording_plan
        ) == 600,

    "400_new_train":
        (
            recording_plan[
                "planned_split"
            ]
            ==
            "train"
        ).sum() == 400,

    "100_new_validation":
        (
            recording_plan[
                "planned_split"
            ]
            ==
            "validation"
        ).sum() == 100,

    "100_new_deployment_test":
        (
            recording_plan[
                "planned_split"
            ]
            ==
            "deployment_test"
        ).sum() == 100,

    "vocabulary_csv_saved":
        VOCAB_CSV.exists(),

    "vocabulary_json_saved":
        VOCAB_JSON.exists(),

    "recording_plan_saved":
        PLAN_PATH.exists(),

    "master_freeze_saved":
        MASTER_FREEZE_PATH.exists(),
}


for name, passed in (
    checks.items()
):

    print(
        f"{name:35s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    checks.values()
):

    failed = [

        key
        for key, value
        in checks.items()
        if not value
    ]


    raise RuntimeError(
        f"L17 failed: {failed}"
    )


# ==================================================================================================
# 28. FINAL FREEZE OUTPUT
# ==================================================================================================

print("\n")
print("=" * 140)
print("L17 — DEPLOYMENT VOCABULARY FREEZE")
print("=" * 140)

print()

print(
    "RESEARCH BENCHMARK"
)

print(
    "  Classes              : 211"
)

print(
    f"  Frozen Test Top-1    : "
    f"{research_top1:.6f}%"
)

print(
    f"  Frozen Test Top-5    : "
    f"{research_top5:.6f}%"
)

print(
    "  Status               : LOCKED"
)

print()

print(
    "DEPLOYMENT EXPERIMENT"
)

print(
    "  Classes              : 10"
)

print(
    "  Accuracy objective   : >=90% validation"
)

print(
    "  Accuracy guaranteed  : NO — must be measured"
)

print()

print("-" * 140)

print(
    "FROZEN 10-SIGN VOCABULARY"
)

print("-" * 140)


for _, row in (
    selected.iterrows()
):

    print(

        f"{int(row['deployment_class_id']):2d}  "
        f"{str(row['gloss']):25s} "

        f"CISLR support="
        f"{int(row['dataset_total_support']):2d}  "

        f"L9 train="
        f"{int(row['train_support']):2d}  "

        f"score="
        f"{float(row['l17_selection_score']):.4f}"
    )


print("-" * 140)

print()

print(
    "L18 NEW DATA PLAN"
)

print(
    f"  New clips/sign       : "
    f"{TARGET_NEW_CLIPS_PER_SIGN}"
)

print(
    f"  Train/sign           : "
    f"{NEW_TRAIN_PER_SIGN}"
)

print(
    f"  Validation/sign      : "
    f"{NEW_VAL_PER_SIGN}"
)

print(
    f"  Deployment test/sign : "
    f"{NEW_TEST_PER_SIGN}"
)

print()

print(
    f"  NEW TRAIN TOTAL      : "
    f"{10 * NEW_TRAIN_PER_SIGN}"
)

print(
    f"  NEW VALIDATION TOTAL : "
    f"{10 * NEW_VAL_PER_SIGN}"
)

print(
    f"  NEW TEST TOTAL       : "
    f"{10 * NEW_TEST_PER_SIGN}"
)

print(
    f"  NEW DATA TOTAL       : "
    f"{10 * TARGET_NEW_CLIPS_PER_SIGN}"
)

print()

print(
    "Existing CISLR L9 TRAIN clips eligible for training:",
    len(
        existing_train
    )
)

print()

print(
    "Vocabulary CSV        :",
    VOCAB_CSV
)

print(
    "Vocabulary JSON       :",
    VOCAB_JSON
)

print(
    "Candidate ranking     :",
    CANDIDATE_PATH
)

print(
    "Existing train subset :",
    EXISTING_TRAIN_PATH
)

print(
    "L18 recording plan    :",
    PLAN_PATH
)

print(
    "L18 protocol          :",
    README_PATH
)

print(
    "Data folders          :",
    DATA_COLLECTION_ROOT
)

print(
    "Summary               :",
    SUMMARY_PATH
)

print(
    "Hash manifest         :",
    HASH_MANIFEST_PATH
)

print(
    "Master freeze         :",
    MASTER_FREEZE_PATH
)

print(
    "Master SHA256         :",
    MASTER_SHA256
)

print()

print(
    "[PASS] L17 COMPLETE AND FROZEN"
)

print(
    "[PASS] 10-sign deployment vocabulary frozen"
)

print(
    "[PASS] L16 test performance NOT used to choose vocabulary"
)

print(
    "[PASS] Research and deployment experiments remain separated"
)

print(
    "[NEXT] L18 = COLLECT / EXPAND DEPLOYMENT DATA"
)

print("=" * 140)

CISLR — L17
10-SIGN RESTRICTED-VOCABULARY DEPLOYMENT EXPERIMENT

Python       : 3.11.15
Random seed  : 42
Vocabulary   : 10 signs
L18 target   : 60 NEW clips/sign

IMPORTANT:
L17 will NOT use L16 test predictions, confusion matrix, or per-class test correctness.

1. REQUIRED ARTIFACT CHECK
dataset.csv                 : PASS
L9 train manifest           : PASS
L9 validation manifest      : PASS
L9 test manifest            : PASS
L9 mapping                  : PASS
L16 final summary           : PASS
CISLR video directory       : PASS

[PASS] Required frozen artifacts exist.

2. VERIFYING L16 RESEARCH FREEZE
Research benchmark classes : 211
Frozen L16 Test Top-1      : 41.232227%
Frozen L16 Test Top-5      : 52.132701%

[PASS] L16 remains historical/frozen.
[PASS] No L16 per-sample predictions loaded.
[PASS] No L16 confusion matrix loaded.
[PASS] No L16 score matrix loaded.

3. ORIGINAL CISLR METADATA
Rows: 7050
Columns: ['uid', 'gloss', 'duration', 'category']
Valid metadata rows: 7049
Uni